# Chess Resources Scraper
Descarga cursos de **modern-chess.com**, vídeos de **YouTube** y un catálogo **manual** (libros/herramientas conocidas).  
El resultado es un CSV unificado listo para usar como DataFrame en el pipeline de recomendación.

## Instalación
```
pip install requests beautifulsoup4 pandas tqdm google-api-python-client
```

## Orden de ejecución
| Celda | Qué hace |
|-------|----------|
| 1 | Imports y configuración |
| 2 | Utilidades comunes (schema, helpers) |
| 3 | Scraper modern-chess.com |
| 4 | Scraper YouTube Data API v3 |
| 5 | Catálogo manual (libros + herramientas) |
| 6 | Merge final → `chess_resources_master.csv` |
| 7 | Exploración rápida del dataset |

In [8]:
import requests
import time
import re
import os
import json
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm

# ── CONFIGURACIÓN ────────────────────────────────────────────────────────
# Ruta donde se guardarán los CSVs (misma carpeta que el notebook por defecto)
OUTPUT_DIR = '.'

# YouTube Data API v3 key (obtener gratis en https://console.cloud.google.com)
# Si no tienes key, la celda de YouTube simplemente se saltará.
YOUTUBE_API_KEY = ''   # <-- pegar tu key aquí

# Delay entre peticiones HTTP para no saturar los servidores
REQUEST_DELAY = 1.5    # segundos

# Máximo de páginas a scrapear en modern-chess (cada página ~10 cursos, ~900 total)
MAX_PAGES_MC = 90

print('✅ Configuración cargada.')

✅ Configuración cargada.


In [9]:
# ── SCHEMA UNIFICADO ─────────────────────────────────────────────────────
# Todas las fuentes producen filas con estas columnas.
# Campos no disponibles en una fuente se dejan vacíos (NaN).
SCHEMA = [
    'resource_id',    # identificador único: fuente__slug
    'source',         # 'modern_chess' | 'youtube' | 'manual'
    'title',          # título completo
    'url',            # enlace directo
    'resource_type',  # 'course' | 'video' | 'playlist' | 'book'
    'format',         # 'video' | 'interactive' | 'text' | 'mixed'
    'course_type',    # 'opening' | 'middlegame' | 'endgame' | 'tactics' | 'general'
    'openings',       # aperturas separadas por '|'
    'color',          # 'White' | 'Black' | 'Both'
    'author',         # nombre del autor / canal
    'author_title',   # 'GM' | 'IM' | 'FM' | 'NM' | ''
    'level_min',      # rating mínimo sugerido (int)
    'level_max',      # rating máximo sugerido (int)
    'level_tier',     # 'beginner' | 'intermediate' | 'advanced' | 'expert'
    'duration_min',   # duración en minutos (float)
    'price_eur',      # precio en euros (0.0 = gratis)
    'is_free',        # 1 | 0
    'has_video',      # 1 | 0
    'has_pgn',        # 1 | 0
    'has_exercises',  # 1 | 0
    'has_spaced_rep', # 1 | 0
    'rating_score',   # valoración 0-5
    'n_reviews',      # número de reseñas
    'views_yt',       # vistas en YouTube
    'pub_date',       # fecha de publicación
    'description',    # primeros 300 chars
    'tags',           # etiquetas adicionales separadas por '|'
]

LEVEL_MAP = {
    'beginner':     (0,    1200),
    'intermediate': (1200, 1800),
    'advanced':     (1800, 2300),
    'expert':       (2300, 3000),
}

OPENING_KEYWORDS = {
    'sicilian':           'Sicilian Defense',
    'french':             'French Defense',
    'caro-kann':          'Caro-Kann Defense',
    "queen's gambit":     "Queen's Gambit",
    "king's indian":      "King's Indian Defense",
    'nimzo':              'Nimzo-Indian Defense',
    'ruy lopez':          'Ruy Lopez',
    'spanish':            'Ruy Lopez',
    'italian':            'Italian Game',
    'london':             'London System',
    'english opening':    'English Opening',
    'grunfeld':           'Gruenfeld Defense',
    'grünfeld':           'Gruenfeld Defense',
    'dutch':              'Dutch Defense',
    'scandinavian':       'Scandinavian Defense',
    'pirc':               'Pirc Defense',
    'benoni':             'Benoni Defense',
    'slav':               'Slav Defense',
    "queen's indian":     "Queen's Indian Defense",
    'catalan':            'Catalan Opening',
    'benko':              'Benko Gambit',
    'trompowsky':         'Trompowsky Attack',
    'vienna':             'Vienna Game',
    'scotch':             'Scotch Game',
    "king's gambit":      "King's Gambit",
    'petroff':            "Petroff's Defense",
    'berlin':             'Berlin Defense',
    'kings indian attack': "King's Indian Attack",
}


def empty_row():
    """Devuelve una fila vacía con todas las columnas del schema."""
    return {col: np.nan for col in SCHEMA}


def make_id(source, slug):
    """Genera un resource_id limpio y único."""
    clean = re.sub(r'[^a-z0-9]', '_', slug.lower())[:40]
    return f"{source}__{clean}"


def extract_openings(text):
    """Detecta nombres de aperturas en texto libre."""
    t = text.lower()
    found = []
    for kw, name in OPENING_KEYWORDS.items():
        if kw in t and name not in found:
            found.append(name)
    return found


def infer_level(text, default_min=1200):
    """Infiere nivel (min, max, tier) a partir de texto libre."""
    t = text.lower()
    m = re.search(r'(\d{3,4})\s*[-]\s*(\d{3,4})', text)
    if m:
        lo, hi = int(m.group(1)), int(m.group(2))
        if 500 <= lo <= 2800 and lo < hi:
            tier = next((k for k, (a, b) in LEVEL_MAP.items() if a <= lo < b), 'intermediate')
            return lo, hi, tier
    if any(w in t for w in ['beginner', 'starter', 'novice']):
        return 0, 1200, 'beginner'
    if any(w in t for w in ['club player', 'intermediate']):
        return 1200, 1800, 'intermediate'
    if any(w in t for w in ['advanced', 'serious player']):
        return 1800, 2300, 'advanced'
    if any(w in t for w in ['master', 'expert', 'titled']):
        return 2300, 3000, 'expert'
    tier = next((k for k, (a, b) in LEVEL_MAP.items() if a <= default_min < b), 'intermediate')
    return default_min, 3000, tier


def iso_duration_to_minutes(duration):
    """Convierte duración ISO 8601 (PT1H30M15S) a minutos."""
    m = re.search(r'(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?', duration or '')
    if not m:
        return np.nan
    h = int(m.group(1) or 0)
    mn = int(m.group(2) or 0)
    s = int(m.group(3) or 0)
    total = h * 60 + mn + s / 60
    return round(total, 1) if total > 0 else np.nan


print('✅ Utilidades cargadas. Schema:', len(SCHEMA), 'columnas.')

✅ Utilidades cargadas. Schema: 27 columnas.


In [10]:
# ── SCRAPER MODERN-CHESS.COM ─────────────────────────────────────────────
# Ejecutar desde tu máquina local.
# El servidor bloquea IPs de datacenter pero permite navegadores normales.

MC_HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/120.0.0.0 Safari/537.36'
    ),
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.9',
}


def parse_mc_card(card):
    """Extrae los datos de una tarjeta de curso de modern-chess."""
    title_tag = card.find('h2')
    if not title_tag:
        return None
    link_tag = card.find('a', href=re.compile(r'/course/'))
    if not link_tag:
        return None

    title = title_tag.get_text(strip=True)
    url = link_tag.get('href', '')
    if not url.startswith('http'):
        url = 'https://www.modern-chess.com' + url

    slug_m = re.search(r'/course/([^/]+)/', url)
    slug = slug_m.group(1) if slug_m else title[:30]
    text = card.get_text(' ')

    # Aperturas desde los links ?opening=
    openings = []
    for a in card.find_all('a', href=re.compile(r'opening=')):
        m = re.search(r'opening=([^&"]+)', a['href'])
        if m:
            openings.append(m.group(1).replace('+', ' '))
    openings = list(dict.fromkeys(openings))  # dedup preservando orden

    # Color
    tl = title.lower()
    color = 'White' if 'for white' in tl else 'Black' if 'for black' in tl else 'Both'

    # Autores
    authors = [a.get_text(strip=True) for a in card.find_all('a', href=re.compile(r'/author/'))]
    author_str = ', '.join(authors)
    author_title = next((t for t in ['GM', 'IM', 'FM', 'NM'] if t in author_str), '')

    # Duración
    duration_min = np.nan
    dur = re.search(r'(\d+(?:\.\d+)?)\s*h(?:\s*and\s*(\d+)\s*min)?|(\d+)\s*min', text)
    if dur:
        if dur.group(3):
            duration_min = float(dur.group(3))
        else:
            duration_min = float(dur.group(1)) * 60 + float(dur.group(2) or 0)

    # Features
    has_video = int('Video Content' in text)
    has_pgn   = int('PGN Download' in text)
    has_ex    = int('Interactive Tests' in text)
    has_boost = int('Memory Booster' in text)

    # Tipo de curso
    if openings or any(w in tl for w in ['opening', 'repertoire', 'defense', 'gambit', 'attack', 'system']):
        ctype = 'opening'
    elif any(w in tl for w in ['middlegame', 'strategy', 'structure', 'pawn', 'positional']):
        ctype = 'middlegame'
    elif any(w in tl for w in ['endgame', 'ending', 'rook ending', 'pawn ending']):
        ctype = 'endgame'
    elif any(w in tl for w in ['tactic', 'puzzle', 'combination', 'calculation']):
        ctype = 'tactics'
    else:
        ctype = 'general'

    # Nivel
    level_min, level_max, tier = infer_level(text)

    # Descripción
    paras = card.find_all('p')
    desc = paras[0].get_text(strip=True)[:300] if paras else ''

    row = empty_row()
    row.update({
        'resource_id':   make_id('mc', slug),
        'source':        'modern_chess',
        'title':         title,
        'url':           url,
        'resource_type': 'course',
        'format':        'video' if has_video else 'interactive',
        'course_type':   ctype,
        'openings':      '|'.join(openings),
        'color':         color,
        'author':        author_str,
        'author_title':  author_title,
        'level_min':     level_min,
        'level_max':     level_max,
        'level_tier':    tier,
        'duration_min':  duration_min,
        'price_eur':     np.nan,
        'is_free':       0,
        'has_video':     has_video,
        'has_pgn':       has_pgn,
        'has_exercises': has_ex,
        'has_spaced_rep':has_boost,
        'description':   desc,
    })
    return row


def scrape_modern_chess(max_pages=MAX_PAGES_MC):
    """
    Descarga todos los cursos de modern-chess.com.
    Guarda progreso cada 10 páginas en src_modern_chess_partial.csv.
    """
    session = requests.Session()
    rows = []
    empty_streak = 0

    for page in tqdm(range(1, max_pages + 1), desc='modern-chess pages'):
        url = (
            f'https://www.modern-chess.com/courses/?pn={page}'
            if page > 1 else
            'https://www.modern-chess.com/courses/'
        )
        try:
            r = session.get(url, headers=MC_HEADERS, timeout=20)
            if r.status_code != 200:
                print(f'  ⚠️  Página {page}: HTTP {r.status_code}')
                empty_streak += 1
                if empty_streak >= 3:
                    break
                continue

            soup = BeautifulSoup(r.text, 'html.parser')
            seen = set()
            found = 0

            # Buscar contenedores de cursos subiendo desde los links /course/
            for link in soup.find_all('a', href=re.compile(r'/course/')):
                container = link
                for _ in range(7):
                    container = container.parent
                    if container is None:
                        break
                    if container.find('h2') and len(container.get_text()) > 100:
                        break
                if container is None:
                    continue
                row = parse_mc_card(container)
                if row and row['url'] not in seen:
                    seen.add(row['url'])
                    rows.append(row)
                    found += 1

            if found == 0:
                empty_streak += 1
                if empty_streak >= 3:
                    print(f'  Fin del catálogo detectado en página {page}.')
                    break
            else:
                empty_streak = 0

            # Guardar progreso cada 10 páginas
            if page % 10 == 0 and rows:
                df_partial = pd.DataFrame(rows).drop_duplicates('resource_id')
                df_partial.to_csv(
                    os.path.join(OUTPUT_DIR, 'src_modern_chess_partial.csv'),
                    index=False
                )
                print(f'  💾 Progreso: {len(df_partial)} cursos (página {page})')

        except Exception as e:
            print(f'  ❌ Página {page}: {e}')

        time.sleep(REQUEST_DELAY + 0.3 * (page % 4))

    df = pd.DataFrame(rows).drop_duplicates('resource_id').reset_index(drop=True)
    out = os.path.join(OUTPUT_DIR, 'src_modern_chess.csv')
    df.to_csv(out, index=False)
    print(f'\n✅ {len(df)} cursos guardados → {out}')
    return df


# ── EJECUTAR ─────────────────────────────────────────────────────────────
df_mc = scrape_modern_chess()
df_mc.head()

modern-chess pages:   0%|          | 0/90 [00:00<?, ?it/s]

  💾 Progreso: 97 cursos (página 10)
  💾 Progreso: 190 cursos (página 20)
  💾 Progreso: 284 cursos (página 30)
  💾 Progreso: 377 cursos (página 40)
  💾 Progreso: 463 cursos (página 50)
  💾 Progreso: 548 cursos (página 60)
  💾 Progreso: 642 cursos (página 70)
  💾 Progreso: 735 cursos (página 80)
  💾 Progreso: 825 cursos (página 90)

✅ 825 cursos guardados → .\src_modern_chess.csv


,resource_id,source,title,url,resource_type,format,course_type,openings,color,author,...,has_video,has_pgn,has_exercises,has_spaced_rep,rating_score,n_reviews,views_yt,pub_date,description,tags
0,mc__dreev_deep_caro_kann_repertoire_for_blac,modern_chess,Dreev Deep Caro-Kann - Repertoire for Black af...,https://www.modern-chess.com/course/dreev-deep...,course,video,opening,Caro-Kann Defense|1.e4,Black,"GMAlexey Dreev, GMPier Luigi Basso",...,1,1,1,1,NaN,NaN,NaN,NaN,"May 5, 2026Caro-Kann Defense1.e4",NaN
1,mc__dragon_for_black,modern_chess,Dragon for BlackPremiumDiscounted,https://www.modern-chess.com/course/dragon-for...,course,video,opening,Sicilian Defense|1.e4,Black,GMSina Movahed,...,1,1,1,1,NaN,NaN,NaN,NaN,"For decades, the Dragon existed in uncertain t...",NaN
2,mc__chebanenko_slav_for_black_part_1,modern_chess,Chebanenko Slav for Black - Part 1,https://www.modern-chess.com/course/chebanenko...,course,video,opening,Slav Defense|1.d4,Black,"GMPier Luigi Basso, GMVladimir Malakhov",...,1,1,1,1,NaN,NaN,NaN,NaN,"May 3, 2026Slav Defense1.d4",NaN
3,mc__inside_your_games_edition_4,modern_chess,Inside Your Games - Edition 4,https://www.modern-chess.com/course/inside-you...,course,video,general,,Both,GMIoannis Papaioannou,...,1,1,1,0,NaN,NaN,NaN,NaN,The fourth installment of GM Ioannis Papaioann...,NaN
4,mc__1_d4_nf6_2_nf3_g6_3_nbd2_deep_understand,modern_chess,1.d4 Nf6 2.Nf3 g6 3.Nbd2 - Deep Understanding,https://www.modern-chess.com/course/1-d4-nf6-2...,course,video,opening,King%27s Indian Defense|Gruenfeld Defense|Pirc...,Both,GMIoannis Papaioannou,...,1,1,0,0,NaN,NaN,NaN,NaN,GM Ioannis Papaioannou presents a complete Whi...,NaN


In [ ]:
# ── SCRAPER YOUTUBE DATA API v3 ──────────────────────────────────────────
# API gratuita: 10.000 unidades/día
# Cómo obtener tu key:
#   1. https://console.cloud.google.com
#   2. Crear proyecto → APIs → 'YouTube Data API v3' → Habilitar
#   3. Credenciales → Crear clave de API
#   4. Pegarla en YOUTUBE_API_KEY en la Celda 1

YT_CHANNELS = {
    'GothamChess':            {'level_min': 0,    'level_max': 1400, 'level_tier': 'beginner'},
    'Hanging Pawns':          {'level_min': 1200, 'level_max': 1900, 'level_tier': 'intermediate'},
    'John Bartholomew':       {'level_min': 1000, 'level_max': 1800, 'level_tier': 'intermediate'},
    'Daniel Naroditsky':      {'level_min': 1600, 'level_max': 2400, 'level_tier': 'advanced'},
    'Saint Louis Chess Club': {'level_min': 1800, 'level_max': 3000, 'level_tier': 'advanced'},
    'PowerPlayChess':         {'level_min': 1400, 'level_max': 2200, 'level_tier': 'intermediate'},
    'Remote Chess Academy':   {'level_min': 1200, 'level_max': 2000, 'level_tier': 'intermediate'},
}

# Playlists específicas de aperturas (playlist_id, canal, apertura, color, level_min)
YT_PLAYLISTS = [
    ('PLBRObh-pknPOdQBSl0v5W8TnFDiEYiKOR', 'GothamChess',       'Sicilian Defense',        'Black', 1000),
    ('PLBRObh-pknPNLhkhj6HpQmNJD_xv2aMVN', 'GothamChess',       'French Defense',          'Black',  800),
    ('PLBRObh-pknPMFWDEsWNLVzRGzxL3vbFz-', 'GothamChess',       "Queen's Gambit",          'Both',  1000),
    ('PLPgbFOFGDcyT1NXLA0KCgI_P7gbfZxmYf', 'Hanging Pawns',     "King's Indian Defense",   'Black', 1400),
    ('PLPgbFOFGDcyT7eXjExHmijbL0yiNz4gLV', 'Hanging Pawns',     'Sicilian Defense',        'Black', 1400),
    ('PLPgbFOFGDcyQ_tILvTq__B4LRvdaTGMqJ', 'Hanging Pawns',     'Caro-Kann Defense',       'Black', 1200),
    ('PLVWaFpMwtaGg2Xc_-s3jqLzpbHtxjEVnD', 'John Bartholomew',  'Caro-Kann Defense',       'Black', 1000),
    ('PLVWaFpMwtaGjG0vD5QinNPfljFBmD5lR1', 'John Bartholomew',  'French Defense',          'Black', 1000),
    ('PLVWaFpMwtaGiBFPFBPWFJkFDAfx83FsEP', 'John Bartholomew',  'Sicilian Defense',        'Black', 1000),
]

# Búsquedas de vídeos por apertura
YT_SEARCHES = [
    ('chess sicilian defense tutorial',    'Sicilian Defense',         'Black'),
    ('chess french defense opening',       'French Defense',           'Black'),
    ('chess caro kann defense tutorial',   'Caro-Kann Defense',        'Black'),
    ('london system opening white chess',  'London System',            'White'),
    ("chess king's indian defense",        "King's Indian Defense",    'Black'),
    ('ruy lopez spanish opening chess',    'Ruy Lopez',                'White'),
    ("queen's gambit declined chess",      "Queen's Gambit Declined",  'Both'),
    ('nimzo indian defense chess tutorial','Nimzo-Indian Defense',     'Black'),
    ('scandinavian defense chess',         'Scandinavian Defense',     'Black'),
    ('italian game chess opening',         'Italian Game',             'White'),
]
YOUTUBE_API_KEY = "TU_YOUTUBE_API_KEY"

def yt_request(endpoint, params):
    params['key'] = YOUTUBE_API_KEY
    r = requests.get(
        f'https://www.googleapis.com/youtube/v3/{endpoint}',
        params=params, timeout=15
    )
    r.raise_for_status()
    return r.json()


def scrape_youtube():
    if not YOUTUBE_API_KEY:
        print('⚠️  YOUTUBE_API_KEY vacía. Actualiza la Celda 1 con tu key.')
        print('   Obtener key gratuita: https://console.cloud.google.com')
        return pd.DataFrame(columns=SCHEMA)

    rows = []

    # 1. Playlists específicas
    print('Descargando playlists...')
    for pl_id, channel, opening, color, lvl_min in tqdm(YT_PLAYLISTS, desc='playlists'):
        try:
            data = yt_request('playlists', {'part': 'snippet,contentDetails', 'id': pl_id})
            if not data.get('items'):
                continue
            pl = data['items'][0]
            title = pl['snippet']['title']
            desc  = pl['snippet'].get('description', '')[:300]
            n_vid = pl['contentDetails']['itemCount']
            _, level_max, tier = infer_level(title + ' ' + desc, lvl_min)

            row = empty_row()
            row.update({
                'resource_id':   make_id('yt', pl_id),
                'source':        'youtube',
                'title':         title,
                'url':           f'https://www.youtube.com/playlist?list={pl_id}',
                'resource_type': 'playlist',
                'format':        'video',
                'course_type':   'opening',
                'openings':      opening,
                'color':         color,
                'author':        channel,
                'level_min':     lvl_min,
                'level_max':     level_max,
                'level_tier':    tier,
                'price_eur':     0.0,
                'is_free':       1,
                'has_video':     1,
                'has_exercises': 0,
                'has_spaced_rep':0,
                'description':   desc,
                'tags':          f'n_videos:{n_vid}',
            })
            rows.append(row)
            time.sleep(0.3)
        except Exception as e:
            print(f'  ⚠️  Playlist {pl_id}: {e}')

    # 2. Búsquedas de vídeos
    print('Buscando vídeos por apertura...')
    for query, opening, color in tqdm(YT_SEARCHES, desc='búsquedas'):
        try:
            results = yt_request('search', {
                'part': 'snippet', 'q': query,
                'type': 'video', 'maxResults': 8,
                'videoDuration': 'medium',
                'relevanceLanguage': 'en',
            })
            video_ids = [item['id']['videoId'] for item in results.get('items', [])]
            if not video_ids:
                continue

            details = yt_request('videos', {
                'part': 'contentDetails,statistics,snippet',
                'id': ','.join(video_ids),
            })
            for item in details.get('items', []):
                vid_id  = item['id']
                sn      = item['snippet']
                stats   = item.get('statistics', {})
                content = item.get('contentDetails', {})
                channel = sn['channelTitle']
                ch_info = YT_CHANNELS.get(channel, {'level_min': 1000, 'level_max': 3000, 'level_tier': 'intermediate'})

                row = empty_row()
                row.update({
                    'resource_id':   make_id('yt', vid_id),
                    'source':        'youtube',
                    'title':         sn['title'],
                    'url':           f'https://www.youtube.com/watch?v={vid_id}',
                    'resource_type': 'video',
                    'format':        'video',
                    'course_type':   'opening',
                    'openings':      opening,
                    'color':         color,
                    'author':        channel,
                    'level_min':     ch_info['level_min'],
                    'level_max':     ch_info['level_max'],
                    'level_tier':    ch_info['level_tier'],
                    'duration_min':  iso_duration_to_minutes(content.get('duration', '')),
                    'price_eur':     0.0,
                    'is_free':       1,
                    'has_video':     1,
                    'views_yt':      int(stats.get('viewCount', 0)),
                    'pub_date':      sn.get('publishedAt', '')[:10],
                    'description':   sn.get('description', '')[:300],
                })
                rows.append(row)
            time.sleep(0.5)
        except Exception as e:
            print(f'  ⚠️  Query "{query}": {e}')

    df = pd.DataFrame(rows).drop_duplicates('resource_id').reset_index(drop=True)
    out = os.path.join(OUTPUT_DIR, 'src_youtube.csv')
    df.to_csv(out, index=False)
    print(f'\n✅ {len(df)} recursos YouTube → {out}')
    return df


df_yt = scrape_youtube()
df_yt.head()

Descargando playlists...


playlists:   0%|          | 0/9 [00:00<?, ?it/s]

Buscando vídeos por apertura...


búsquedas:   0%|          | 0/10 [00:00<?, ?it/s]


✅ 80 recursos YouTube → .\src_youtube.csv


,resource_id,source,title,url,resource_type,format,course_type,openings,color,author,...,has_video,has_pgn,has_exercises,has_spaced_rep,rating_score,n_reviews,views_yt,pub_date,description,tags
0,yt__qm4e7g2ruki,youtube,The Sicilian Defense | 10-Minute Chess Openings,https://www.youtube.com/watch?v=qM4e7g2RukI,video,video,opening,Sicilian Defense,Black,GothamChess,...,1,NaN,NaN,NaN,NaN,NaN,3503268,2020-06-11,➡️ Get My Chess Courses: https://www.chessly....,NaN
1,yt__0iayykxh3zc,youtube,DOMINATE as Black with the Sicilian Defense,https://www.youtube.com/watch?v=0iAyyKxh3zc,video,video,opening,Sicilian Defense,Black,ChessPage1,...,1,NaN,NaN,NaN,NaN,NaN,710304,2025-07-26,Try chessreps for FREE using my link so they k...,NaN
2,yt__2miollk8dii,youtube,Sicilian Defense ALL Variations Explained in 1...,https://www.youtube.com/watch?v=2miolLK8DiI,video,video,opening,Sicilian Defense,Black,Remote Chess Academy,...,1,NaN,NaN,NaN,NaN,NaN,780174,2024-01-26,Learn 3 Simple Rules To Reach 2000+ ELO Rating...,NaN
3,yt__6msbe_cahf4,youtube,BEST Chess Opening for Black: Sicilian Defense...,https://www.youtube.com/watch?v=6msbe_CAhf4,video,video,opening,Sicilian Defense,Black,Chess Talk,...,1,NaN,NaN,NaN,NaN,NaN,2293896,2018-03-03,Find out the Best Chess Opening Strategy for B...,NaN
4,yt__n2vdfnrwrfa,youtube,HOW TO PLAY SICILIAN DEFENSE! Opening explained,https://www.youtube.com/watch?v=n2vdFNrWrFA,video,video,opening,Sicilian Defense,Black,Biyaherong Chess Coach,...,1,NaN,NaN,NaN,NaN,NaN,136935,2022-09-13,For more videos\nWesley So Notable Game\nhttps...,NaN


In [12]:
# ── CATÁLOGO MANUAL ──────────────────────────────────────────────────────
# Libros clásicos, herramientas online y cursos premium conocidos.
# Añade filas aquí cuando encuentres nuevos recursos relevantes.

MANUAL = [
    # ── LIBROS ───────────────────────────────────────────────────────────
    dict(title='My System',                    author='Nimzowitsch',
         resource_type='book', format='text',  course_type='middlegame',
         openings='', color='Both',            level_min=1400, level_max=2200,
         price_eur=20.0, is_free=0, has_video=0, has_exercises=0,
         url='https://www.amazon.com/s?k=My+System+Nimzowitsch',
         description='Fundamental positional chess: blockade, prophylaxis, overprotection.'),
    dict(title='Chess Fundamentals',            author='Capablanca',
         resource_type='book', format='text',  course_type='general',
         openings='', color='Both',            level_min=0, level_max=1400,
         price_eur=0.0, is_free=1, has_video=0, has_exercises=0,
         url='https://www.gutenberg.org/ebooks/33370',
         description='Essential principles for beginners. Free on Project Gutenberg.'),
    dict(title="Dvoretsky's Endgame Manual",    author='Dvoretsky',
         resource_type='book', format='text',  course_type='endgame',
         openings='', color='Both',            level_min=2000, level_max=3000,
         price_eur=35.0, is_free=0, has_video=0, has_exercises=1,
         url='https://www.amazon.com/s?k=Dvoretsky+Endgame+Manual',
         description='Definitive endgame reference for advanced and master-level players.'),
    dict(title="Silman's Complete Endgame Course", author='Silman',
         resource_type='book', format='text',  course_type='endgame',
         openings='', color='Both',            level_min=800, level_max=2200,
         price_eur=25.0, is_free=0, has_video=0, has_exercises=1,
         url='https://www.amazon.com/s?k=Silman+Complete+Endgame',
         description='Endgame curriculum organized by rating level from beginner to master.'),
    dict(title='Winning Chess Tactics',          author='Seirawan',
         resource_type='book', format='text',  course_type='tactics',
         openings='', color='Both',            level_min=800, level_max=1600,
         price_eur=20.0, is_free=0, has_video=0, has_exercises=1,
         url='https://www.amazon.com/s?k=Winning+Chess+Tactics+Seirawan',
         description='Systematic introduction to forks, pins, skewers and discovered attacks.'),
    dict(title='Understanding Chess Middlegames',  author='Nunn',
         resource_type='book', format='text',  course_type='middlegame',
         openings='', color='Both',            level_min=1800, level_max=2600,
         price_eur=28.0, is_free=0, has_video=0, has_exercises=1,
         url='https://www.amazon.com/s?k=Understanding+Chess+Middlegames+Nunn',
         description='100 middlegame principles with examples from GM games.'),
    dict(title='The Sicilian Najdorf',             author='Kasparov',
         resource_type='book', format='text',  course_type='opening',
         openings='Sicilian Defense', color='Black', level_min=1800, level_max=3000,
         price_eur=30.0, is_free=0, has_video=0, has_exercises=0,
         url='https://www.amazon.com/s?k=Kasparov+Sicilian+Najdorf',
         description="Kasparov's definitive analysis of the Najdorf Sicilian."),
    dict(title='Play the London System',           author='Sverre Johnsen',
         resource_type='book', format='text',  course_type='opening',
         openings='London System', color='White', level_min=1000, level_max=1800,
         price_eur=22.0, is_free=0, has_video=0, has_exercises=0,
         url='https://www.amazon.com/s?k=Play+London+System+Johnsen',
         description='Complete practical guide to the London System for White.'),
    dict(title='Pump Up Your Rating',              author='Axel Smith',
         resource_type='book', format='text',  course_type='general',
         openings='', color='Both',            level_min=1600, level_max=2200,
         price_eur=24.0, is_free=0, has_video=0, has_exercises=1,
         url='https://www.amazon.com/s?k=Pump+Up+Your+Rating+Axel+Smith',
         description='Training methods and study techniques for serious improvement.'),
    # ── HERRAMIENTAS ONLINE ──────────────────────────────────────────────
    dict(title='Magnus Carlsen MasterClass',     author='Magnus Carlsen',
         resource_type='course', format='video', course_type='general',
         openings='', color='Both',             level_min=1200, level_max=3000,
         price_eur=90.0, is_free=0, has_video=1, has_exercises=0,
         url='https://www.masterclass.com/classes/magnus-carlsen-teaches-chess',
         description='Strategy, opening preparation and mindset by the world champion.'),
    dict(title='Lichess Opening Explorer',        author='Lichess',
         resource_type='course', format='interactive', course_type='opening',
         openings='', color='Both',             level_min=0, level_max=3000,
         price_eur=0.0, is_free=1, has_video=0, has_exercises=1,
         url='https://lichess.org/analysis',
         description='Free opening explorer with master game database and engine analysis.'),
    dict(title='ChessTempo Tactics Trainer',      author='ChessTempo',
         resource_type='course', format='interactive', course_type='tactics',
         openings='', color='Both',             level_min=800, level_max=3000,
         price_eur=0.0, is_free=1, has_video=0, has_exercises=1,
         url='https://chesstempo.com',
         description='Adaptive tactics training with spaced repetition and performance tracking.'),
    dict(title='Chess.com Lessons — Beginners',   author='Chess.com',
         resource_type='course', format='interactive', course_type='general',
         openings='', color='Both',             level_min=0, level_max=1200,
         price_eur=0.0, is_free=1, has_video=1, has_exercises=1,
         url='https://www.chess.com/learn-how-to-play-chess',
         description='Free introductory lessons covering opening principles, tactics and endgames.'),
]


def build_manual_dataset():
    rows = []
    for res in MANUAL:
        slug = re.sub(r'\W+', '_', res['title'].lower())[:40]
        tier = next(
            (t for t, (a, b) in LEVEL_MAP.items() if a <= res.get('level_min', 1200) < b),
            'intermediate'
        )
        row = empty_row()
        row.update({
            'resource_id':   make_id('man', slug),
            'source':        'manual',
            'title':         res['title'],
            'url':           res.get('url', ''),
            'resource_type': res.get('resource_type', 'book'),
            'format':        res.get('format', 'text'),
            'course_type':   res.get('course_type', 'general'),
            'openings':      res.get('openings', ''),
            'color':         res.get('color', 'Both'),
            'author':        res.get('author', ''),
            'author_title':  '',
            'level_min':     res.get('level_min', 1200),
            'level_max':     res.get('level_max', 3000),
            'level_tier':    tier,
            'price_eur':     res.get('price_eur', np.nan),
            'is_free':       res.get('is_free', 0),
            'has_video':     res.get('has_video', 0),
            'has_pgn':       res.get('has_pgn', 0),
            'has_exercises': res.get('has_exercises', 0),
            'has_spaced_rep':res.get('has_spaced_rep', 0),
            'description':   res.get('description', ''),
        })
        rows.append(row)

    df = pd.DataFrame(rows)[SCHEMA]
    out = os.path.join(OUTPUT_DIR, 'src_manual.csv')
    df.to_csv(out, index=False)
    print(f'✅ {len(df)} recursos manuales → {out}')
    return df


df_manual = build_manual_dataset()
df_manual

✅ 13 recursos manuales → .\src_manual.csv


,resource_id,source,title,url,resource_type,format,course_type,openings,color,author,...,has_video,has_pgn,has_exercises,has_spaced_rep,rating_score,n_reviews,views_yt,pub_date,description,tags
0,man__my_system,manual,My System,https://www.amazon.com/s?k=My+System+Nimzowitsch,book,text,middlegame,,Both,Nimzowitsch,...,0,0,0,0,NaN,NaN,NaN,NaN,"Fundamental positional chess: blockade, prophy...",NaN
1,man__chess_fundamentals,manual,Chess Fundamentals,https://www.gutenberg.org/ebooks/33370,book,text,general,,Both,Capablanca,...,0,0,0,0,NaN,NaN,NaN,NaN,Essential principles for beginners. Free on Pr...,NaN
2,man__dvoretsky_s_endgame_manual,manual,Dvoretsky's Endgame Manual,https://www.amazon.com/s?k=Dvoretsky+Endgame+M...,book,text,endgame,,Both,Dvoretsky,...,0,0,1,0,NaN,NaN,NaN,NaN,Definitive endgame reference for advanced and ...,NaN
3,man__silman_s_complete_endgame_course,manual,Silman's Complete Endgame Course,https://www.amazon.com/s?k=Silman+Complete+End...,book,text,endgame,,Both,Silman,...,0,0,1,0,NaN,NaN,NaN,NaN,Endgame curriculum organized by rating level f...,NaN
4,man__winning_chess_tactics,manual,Winning Chess Tactics,https://www.amazon.com/s?k=Winning+Chess+Tacti...,book,text,tactics,,Both,Seirawan,...,0,0,1,0,NaN,NaN,NaN,NaN,"Systematic introduction to forks, pins, skewer...",NaN
5,man__understanding_chess_middlegames,manual,Understanding Chess Middlegames,https://www.amazon.com/s?k=Understanding+Chess...,book,text,middlegame,,Both,Nunn,...,0,0,1,0,NaN,NaN,NaN,NaN,100 middlegame principles with examples from G...,NaN
6,man__the_sicilian_najdorf,manual,The Sicilian Najdorf,https://www.amazon.com/s?k=Kasparov+Sicilian+N...,book,text,opening,Sicilian Defense,Black,Kasparov,...,0,0,0,0,NaN,NaN,NaN,NaN,Kasparov's definitive analysis of the Najdorf ...,NaN
7,man__play_the_london_system,manual,Play the London System,https://www.amazon.com/s?k=Play+London+System+...,book,text,opening,London System,White,Sverre Johnsen,...,0,0,0,0,NaN,NaN,NaN,NaN,Complete practical guide to the London System ...,NaN
8,man__pump_up_your_rating,manual,Pump Up Your Rating,https://www.amazon.com/s?k=Pump+Up+Your+Rating...,book,text,general,,Both,Axel Smith,...,0,0,1,0,NaN,NaN,NaN,NaN,Training methods and study techniques for seri...,NaN
9,man__magnus_carlsen_masterclass,manual,Magnus Carlsen MasterClass,https://www.masterclass.com/classes/magnus-car...,course,video,general,,Both,Magnus Carlsen,...,1,0,0,0,NaN,NaN,NaN,NaN,"Strategy, opening preparation and mindset by t...",NaN


In [13]:
# ── MERGE FINAL ──────────────────────────────────────────────────────────
# Une todos los CSVs de fuentes en un master dataset.

def merge_all_sources():
    source_files = {
        'modern-chess': os.path.join(OUTPUT_DIR, 'src_modern_chess.csv'),
        'youtube':       os.path.join(OUTPUT_DIR, 'src_youtube.csv'),
        'manual':        os.path.join(OUTPUT_DIR, 'src_manual.csv'),
    }

    dfs = []
    for name, path in source_files.items():
        if not os.path.exists(path):
            print(f'  ⚠️  {name}: archivo no encontrado ({path})')
            continue
        df_src = pd.read_csv(path)
        # Asegurar que todas las columnas del schema existen
        for col in SCHEMA:
            if col not in df_src.columns:
                df_src[col] = np.nan
        dfs.append(df_src[SCHEMA])
        print(f'  📥 {name}: {len(df_src)} registros')

    if not dfs:
        print('❌ No hay fuentes disponibles para mergear.')
        return pd.DataFrame()

    df = pd.concat(dfs, ignore_index=True).drop_duplicates('resource_id')

    # Normalizar level_tier donde falte
    def fix_tier(row):
        if pd.notna(row['level_tier']):
            return row['level_tier']
        lvl = row['level_min'] if pd.notna(row['level_min']) else 1200
        return next((t for t, (a, b) in LEVEL_MAP.items() if a <= lvl < b), 'intermediate')
    df['level_tier'] = df.apply(fix_tier, axis=1)

    # Guardar
    out = os.path.join(OUTPUT_DIR, 'chess_resources_master.csv')
    df.to_csv(out, index=False)

    print(f'\n✅ MASTER DATASET: {len(df)} recursos únicos → {out}')
    print('\nPor fuente:')
    print(df['source'].value_counts().to_string())
    print('\nPor tipo de recurso:')
    print(df['resource_type'].value_counts().to_string())
    print('\nPor tipo de curso:')
    print(df['course_type'].value_counts().to_string())
    print('\nPor nivel:')
    print(df['level_tier'].value_counts().to_string())
    return df


df_master = merge_all_sources()
df_master.head(10)

  📥 modern-chess: 825 registros
  📥 youtube: 80 registros
  📥 manual: 13 registros

✅ MASTER DATASET: 918 recursos únicos → .\chess_resources_master.csv

Por fuente:
source
modern_chess    825
youtube          80
manual           13

Por tipo de recurso:
resource_type
course    829
video      80
book        9

Por tipo de curso:
course_type
opening       759
general        74
endgame        34
middlegame     29
tactics        22

Por nivel:
level_tier
intermediate    703
expert          160
advanced         37
beginner         18


,resource_id,source,title,url,resource_type,format,course_type,openings,color,author,...,has_video,has_pgn,has_exercises,has_spaced_rep,rating_score,n_reviews,views_yt,pub_date,description,tags
0,mc__dreev_deep_caro_kann_repertoire_for_blac,modern_chess,Dreev Deep Caro-Kann - Repertoire for Black af...,https://www.modern-chess.com/course/dreev-deep...,course,video,opening,Caro-Kann Defense|1.e4,Black,"GMAlexey Dreev, GMPier Luigi Basso",...,1,1.0,1.0,1.0,NaN,NaN,NaN,NaN,"May 5, 2026Caro-Kann Defense1.e4",NaN
1,mc__dragon_for_black,modern_chess,Dragon for BlackPremiumDiscounted,https://www.modern-chess.com/course/dragon-for...,course,video,opening,Sicilian Defense|1.e4,Black,GMSina Movahed,...,1,1.0,1.0,1.0,NaN,NaN,NaN,NaN,"For decades, the Dragon existed in uncertain t...",NaN
2,mc__chebanenko_slav_for_black_part_1,modern_chess,Chebanenko Slav for Black - Part 1,https://www.modern-chess.com/course/chebanenko...,course,video,opening,Slav Defense|1.d4,Black,"GMPier Luigi Basso, GMVladimir Malakhov",...,1,1.0,1.0,1.0,NaN,NaN,NaN,NaN,"May 3, 2026Slav Defense1.d4",NaN
3,mc__inside_your_games_edition_4,modern_chess,Inside Your Games - Edition 4,https://www.modern-chess.com/course/inside-you...,course,video,general,NaN,Both,GMIoannis Papaioannou,...,1,1.0,1.0,0.0,NaN,NaN,NaN,NaN,The fourth installment of GM Ioannis Papaioann...,NaN
4,mc__1_d4_nf6_2_nf3_g6_3_nbd2_deep_understand,modern_chess,1.d4 Nf6 2.Nf3 g6 3.Nbd2 - Deep Understanding,https://www.modern-chess.com/course/1-d4-nf6-2...,course,video,opening,King%27s Indian Defense|Gruenfeld Defense|Pirc...,Both,GMIoannis Papaioannou,...,1,1.0,0.0,0.0,NaN,NaN,NaN,NaN,GM Ioannis Papaioannou presents a complete Whi...,NaN
5,mc__french_steinitz_with_rb8_repertoire_for_,modern_chess,French Steinitz with ...Rb8 - Repertoire for B...,https://www.modern-chess.com/course/french-ste...,course,video,opening,French Defense|1.e4,Black,"GMPier Luigi Basso, GMSzymon Gumularz",...,1,1.0,1.0,1.0,NaN,NaN,NaN,NaN,"May 1, 2026French Defense1.e4",NaN
6,mc__kings_indian_defense_repertoire_for_blac,modern_chess,King's Indian Defense: Repertoire for Black - ...,https://www.modern-chess.com/course/kings-indi...,course,video,opening,King%27s Indian Defense,Black,"GMAndrea Stella, GMBaadur Jobava",...,1,1.0,1.0,1.0,NaN,NaN,NaN,NaN,"April 30, 2026King's Indian Defense",NaN
7,mc__french_defense_middlegame_understanding_,modern_chess,French Defense - Middlegame Understanding: Ope...,https://www.modern-chess.com/course/french-def...,course,video,opening,NaN,Both,GMDavorin Kuljasevic,...,1,1.0,1.0,0.0,NaN,NaN,NaN,NaN,The second part of GM Davorin Kuljasevic's Und...,NaN
8,mc__1_nf3_nf6_2_g3_g6_3_c4_a_flexible_repert,modern_chess,1.Nf3 Nf6 2.g3 g6 3.c4: A Flexible Repertoire ...,https://www.modern-chess.com/course/1-nf3-nf6-...,course,video,opening,Reti Opening|1.Nf3,White,GMSina Movahed,...,1,1.0,1.0,1.0,NaN,NaN,NaN,NaN,"This flexible system, anchored by the move ord...",NaN
9,mc__sveshnikov_sicilian_for_white_top_level_,modern_chess,Sveshnikov Sicilian for White - Top-Level Repe...,https://www.modern-chess.com/course/sveshnikov...,course,video,opening,Sicilian Defense|1.e4,White,"IMDragos Ceres, GMJose Martinez Alcantara",...,1,1.0,1.0,1.0,NaN,NaN,NaN,NaN,"April 28, 2026Sicilian Defense1.e4",NaN


In [14]:
# ── EXPLORACIÓN DEL DATASET ──────────────────────────────────────────────

if 'df_master' not in dir() or df_master is None or df_master.empty:
    master_path = os.path.join(OUTPUT_DIR, 'chess_resources_master.csv')
    df_master = pd.read_csv(master_path) if os.path.exists(master_path) else pd.DataFrame()

if df_master.empty:
    print('❌ No hay datos. Ejecuta las celdas anteriores primero.')
else:
    print('=== RESUMEN GENERAL ===')
    print(f'Total recursos:  {len(df_master)}')
    print(f'Columnas:        {len(df_master.columns)}')
    print(f'Nulos por col:\n{df_master.isnull().mean().round(2).to_string()}')

    print('\n=== TOP 10 APERTURAS MÁS CUBIERTAS ===')
    all_ops = (
        pd.Series([
            op.strip()
            for ops in df_master['openings'].fillna('').str.split('|')
            for op in ops if op.strip()
        ])
        .value_counts()
        .head(10)
    )
    print(all_ops.to_string())

    print('\n=== RECURSOS POR NIVEL Y TIPO ===')
    pivot = df_master.pivot_table(
        index='level_tier', columns='course_type',
        values='resource_id', aggfunc='count', fill_value=0
    )
    level_order = ['beginner', 'intermediate', 'advanced', 'expert']
    pivot = pivot.reindex([l for l in level_order if l in pivot.index])
    print(pivot.to_string())

    print('\n=== RECURSOS GRATUITOS vs PAGO ===')
    print(df_master['is_free'].map({1: 'Gratis', 0: 'Pago'}).value_counts().to_string())

    print('\n=== MUESTRA DE CURSOS DE APERTURA (nivel intermediate) ===')
    sample = df_master[
        (df_master['course_type'] == 'opening') &
        (df_master['level_tier'] == 'intermediate')
    ][['title', 'source', 'openings', 'color', 'url']].head(8)
    pd.set_option('display.max_colwidth', 50)
    print(sample.to_string(index=False))

=== RESUMEN GENERAL ===
Total recursos:  918
Columnas:        27
Nulos por col:
resource_id       0.00
source            0.00
title             0.00
url               0.00
resource_type     0.00
format            0.00
course_type       0.00
openings          0.20
color             0.00
author            0.00
author_title      0.16
level_min         0.00
level_max         0.00
level_tier        0.00
duration_min      0.37
price_eur         0.90
is_free           0.00
has_video         0.00
has_pgn           0.09
has_exercises     0.09
has_spaced_rep    0.09
rating_score      1.00
n_reviews         1.00
views_yt          0.91
pub_date          0.91
description       0.00
tags              1.00

=== TOP 10 APERTURAS MÁS CUBIERTAS ===
1.e4                         315
1.d4                         256
Sicilian Defense             118
French Defense                56
Caro-Kann Defense             53
Ruy Lopez                     52
Queen%27s Gambit Declined     40
1.Nf3                       

course_type   endgame  general  middlegame  opening  tactics
level_tier                                                  
beginner            2        4           0       10        2
intermediate       21       44          14      618        6
advanced            1        9           5       15        7
expert             10       17          10      116        7

=== RECURSOS GRATUITOS vs PAGO ===
is_free
Pago      834
Gratis     84

=== MUESTRA DE CURSOS DE APERTURA (nivel intermediate) ===
                                                                         title       source                       openings color                                                                                                                  url
               Dreev Deep Caro-Kann - Repertoire for Black after 1.e4 c6 2.Nf3 modern_chess         Caro-Kann Defense|1.e4 Black             https://www.modern-chess.com/course/dreev-deep-caro-kann-repertoire-for-black-after-1-e4-c6-2-nf3/58916/
          

In [15]:
import os
import glob

# Buscar chess_resources_master.csv en el disco
for drive in ["C:\\Users\\Eneko\\Desktop", "C:\\Users\\Eneko\\Documents"]:
    results = glob.glob(os.path.join(drive, "**", "chess_resources_master.csv"), recursive=True)
    for r in results:
        print(f"✅ Encontrado: {r}")

✅ Encontrado: C:\Users\Eneko\Desktop\Ejercicios Data\Proyectos\Proyecto ML\chess_resources_master.csv


In [ ]:
# ── CELDA ÚNICA — SCRAPING BEGINNER COMPLETO ─────────────────────────────
import requests, re, time, json
import numpy as np
import pandas as pd
from googleapiclient.discovery import build

HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/120.0.0.0 Safari/537.36'
    )
}

BEGINNER_SIGNALS = [
    'beginner', 'basics', 'fundamental', 'starter', 'easy',
    'simple', 'dummies', '101', 'quick start', 'introduction',
    'intro', 'learn chess', 'first steps', 'new to chess',
    'getting started', 'kids', 'children', 'complete beginner',
]

SCHEMA_COLS = [
    'resource_id', 'source', 'title', 'url', 'resource_type',
    'format', 'course_type', 'openings', 'color', 'author',
    'author_title', 'level_min', 'level_max', 'level_tier',
    'duration_min', 'price_eur', 'is_free', 'has_video',
    'has_pgn', 'has_exercises', 'has_spaced_rep', 'rating_score',
    'n_reviews', 'views_yt', 'pub_date', 'description', 'tags',
    'level_tier_original', 'level_tier_new', 'level_tier_source',
    'score_up', 'score_down', 'score_anchor',
]

def empty_row():
    return {col: np.nan for col in SCHEMA_COLS}

def make_id(source, slug):
    clean = re.sub(r'[^a-z0-9]', '_', slug.lower())[:40]
    return f"{source}__{clean}"

def is_beginner(text: str) -> bool:
    return any(s in text.lower() for s in BEGINNER_SIGNALS)

def iso_to_minutes(duration: str) -> float:
    m = re.search(r'(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?', duration or '')
    if not m:
        return np.nan
    total = int(m.group(1) or 0)*60 + int(m.group(2) or 0) + int(m.group(3) or 0)/60
    return round(total, 1) if total > 0 else np.nan

def base_beginner_fields():
    return {
        'level_min': 0, 'level_max': 1200,
        'level_tier': 'beginner',
        'level_tier_original': 'beginner',
        'level_tier_new': 'beginner',
        'level_tier_source': 'manual_label',
        'score_up': 0, 'score_down': 2, 'score_anchor': 0,
        'is_free': 1,
    }

# ── FUENTE 1: LICHESS ────────────────────────────────────────────────────
def scrape_lichess() -> pd.DataFrame:
    queries = ['beginner', 'basics', 'fundamentals', 'learn chess', 'starter']
    rows, seen = [], set()

    for q in queries:
        try:
            r = requests.get(
                'https://lichess.org/api/study/search',
                params={'q': q, 'nb': 50},
                headers={**HEADERS, 'Accept': 'application/x-ndjson'},
                timeout=10
            )
            for line in r.text.strip().split('\n'):
                if not line:
                    continue
                try:
                    study = json.loads(line)
                except:
                    continue

                title = study.get('name', '')
                sid   = study.get('id', '')

                if sid in seen or not is_beginner(title):
                    continue
                seen.add(sid)

                row = empty_row()
                row.update(base_beginner_fields())
                row.update({
                    'resource_id':   make_id('lichess', title),
                    'source':        'lichess',
                    'title':         title,
                    'url':           f"https://lichess.org/study/{sid}",
                    'resource_type': 'course',
                    'format':        'interactive',
                    'course_type':   'general',
                    'color':         'Both',
                    'has_pgn':       1,
                    'has_exercises': 1,
                    'description':   study.get('description', '')[:300],
                })
                rows.append(row)

            time.sleep(1.5)
            print(f"  ✓ Lichess '{q}': {len(seen)} únicos acumulados")

        except Exception as e:
            print(f"  ⚠️  Lichess '{q}': {e}")

    df = pd.DataFrame(rows).drop_duplicates('resource_id') if rows \
         else pd.DataFrame(columns=SCHEMA_COLS)
    print(f"✅ Lichess total: {len(df)}\n")
    return df


# ── FUENTE 2: YOUTUBE ────────────────────────────────────────────────────
YT_QUERIES = [
    ('chess for absolute beginners',        'general',    'Both'),
    ('chess basics tutorial complete',      'general',    'Both'),
    ('best chess openings beginners white', 'opening',    'White'),
    ('best chess openings beginners black', 'opening',    'Black'),
    ('chess tactics beginners easy',        'tactics',    'Both'),
    ('chess endgame beginners simple',      'endgame',    'Both'),
    ('how to play chess step by step',      'general',    'Both'),
    ('simple chess repertoire 1200',        'opening',    'Both'),
    ('chess dummies quick guide',           'general',    'Both'),
    ('chess middlegame basics easy',        'middlegame', 'Both'),
    ('learn chess fast beginners',          'general',    'Both'),
    ('chess opening principles beginners',  'opening',    'Both'),
]
YOUTUBE_API_KEY = "TU_YOUTUBE_TOKEN_AQUI"
def scrape_youtube() -> pd.DataFrame:
    yt   = build('youtube', 'v3', developerKey=YOUTUBE_API_KEY)
    rows, seen = [], set()

    for query, course_type, color in YT_QUERIES:
        try:
            search = yt.search().list(
                q=query, part='snippet', type='video',
                maxResults=20, relevanceLanguage='en',
                videoDuration='medium',
            ).execute()

            ids = [i['id']['videoId'] for i in search.get('items', [])]
            if not ids:
                continue

            stats_resp = yt.videos().list(
                part='statistics,contentDetails',
                id=','.join(ids)
            ).execute()
            stats_map = {s['id']: s for s in stats_resp.get('items', [])}

            added = 0
            for item in search.get('items', []):
                vid_id  = item['id']['videoId']
                snippet = item['snippet']
                title   = snippet.get('title', '')
                desc    = snippet.get('description', '')[:300]

                if vid_id in seen:
                    continue
                if not is_beginner(f"{title} {desc}"):
                    continue
                seen.add(vid_id)
                added += 1

                st = stats_map.get(vid_id, {}).get('statistics', {})
                ct = stats_map.get(vid_id, {}).get('contentDetails', {})

                row = empty_row()
                row.update(base_beginner_fields())
                row.update({
                    'resource_id':   make_id('yt_beg', title),
                    'source':        'youtube',
                    'title':         title,
                    'url':           f'https://youtube.com/watch?v={vid_id}',
                    'resource_type': 'video',
                    'format':        'video',
                    'course_type':   course_type,
                    'color':         color,
                    'author':        snippet.get('channelTitle', ''),
                    'has_video':     1,
                    'duration_min':  iso_to_minutes(ct.get('duration', '')),
                    'views_yt':      int(st.get('viewCount', 0)),
                    'pub_date':      snippet.get('publishedAt', '')[:10],
                    'description':   desc,
                })
                rows.append(row)

            time.sleep(1)
            print(f"  ✓ '{query}': +{added} (total {len(seen)})")

        except Exception as e:
            print(f"  ⚠️  YT '{query}': {e}")

    df = pd.DataFrame(rows).drop_duplicates('resource_id') if rows \
         else pd.DataFrame(columns=SCHEMA_COLS)
    print(f"✅ YouTube total: {len(df)}\n")
    return df


# ── MERGE FINAL ──────────────────────────────────────────────────────────
def build_final_dataset() -> pd.DataFrame:
    print("=" * 50)
    print("▶ SCRAPING BEGINNER — INICIO")
    print("=" * 50)

    print("\n[1/2] Lichess Studies...")
    df_lichess = scrape_lichess()

    print("[2/2] YouTube...")
    df_yt = scrape_youtube()

    df_new = pd.concat([df_lichess, df_yt], ignore_index=True) \
               .drop_duplicates('resource_id')

    df_base = pd.read_csv('chess_resources_3tier.csv')

    # Alinear columnas
    for col in SCHEMA_COLS:
        if col not in df_base.columns:
            df_base[col] = np.nan
        if col not in df_new.columns:
            df_new[col] = np.nan

    df_new_only = df_new[~df_new['resource_id'].isin(df_base['resource_id'])]
    df_final    = pd.concat([df_base, df_new_only], ignore_index=True)

    # ── REPORTE ──────────────────────────────────────────────────────────
    order   = ['beginner', 'intermediate', 'advanced']
    targets = {'beginner': 20, 'intermediate': 50, 'advanced': 30}
    before  = {'beginner': 92, 'intermediate': 697, 'advanced': 219}
    dist     = df_final['level_tier'].value_counts().reindex(order)
    dist_pct = df_final['level_tier'].value_counts(normalize=True).reindex(order) * 100

    print("\n" + "=" * 62)
    print("DATASET FINAL")
    print("=" * 62)
    print(f"{'Nivel':<15} {'Antes':>7} {'Después':>8} {'%':>7}  {'Obj':>5}  Estado")
    print("-" * 62)
    for tier in order:
        n   = dist[tier]
        pct = dist_pct[tier]
        tgt = targets[tier]
        st  = "✅" if pct >= tgt else "⚠️  BAJO"
        print(f"{tier:<15} {before[tier]:>7} {n:>8} {pct:>6.1f}%  {tgt:>4}%  {st}")
    print("-" * 62)
    print(f"Nuevos recursos añadidos: {len(df_new_only)}")
    print(f"Total final:              {len(df_final)}")

    df_final.to_csv('chess_resources_final.csv', index=False)
    df_new_only.to_csv('src_beginner_new.csv', index=False)
    print("\n✅ chess_resources_final.csv")
    print("✅ src_beginner_new.csv")
    return df_final


df_final = build_final_dataset()

In [ ]:
# CELDA DE SCRAPPING PARA AÑADIR MAS RECURSOS A BEGINNER

# ══════════════════════════════════════════════════════════════════════════
# CATÁLOGO MANUAL — 200+ Recursos Beginner/Intermediate
# Genera recursos compatibles con chess_resources_master.csv
# ══════════════════════════════════════════════════════════════════════════

import re
import numpy as np
import pandas as pd
import os

OUTPUT_PATH = "beginner_resources.csv"
MASTER_PATH = "chess_resources_master.csv"

def make_id(source_prefix, title):
    clean = re.sub(r'[^a-z0-9]+', '_', title.lower()).strip('_')[:45]
    return f"{source_prefix}__{clean}"

# ── SCHEMA BASE ───────────────────────────────────────────────────────────
def r(source, title, url, resource_type, format_, course_type, openings,
      color, author, author_title, level_min, level_max, level_tier,
      duration_min, price_eur, is_free, has_video, has_pgn, has_exercises,
      has_spaced_rep, rating_score, n_reviews, description):
    prefix = {'book': 'man', 'course': source[:3],
              'video': 'yt', 'playlist': 'yt'}.get(resource_type, 'man')
    return {
        'resource_id':    make_id(prefix, title),
        'source':         source,
        'title':          title,
        'url':            url,
        'resource_type':  resource_type,
        'format':         format_,
        'course_type':    course_type,
        'openings':       openings,
        'color':          color,
        'author':         author,
        'author_title':   author_title,
        'level_min':      level_min,
        'level_max':      level_max,
        'level_tier':     level_tier,
        'duration_min':   duration_min,
        'price_eur':      price_eur,
        'is_free':        is_free,
        'has_video':      has_video,
        'has_pgn':        has_pgn,
        'has_exercises':  has_exercises,
        'has_spaced_rep': has_spaced_rep,
        'rating_score':   rating_score,
        'n_reviews':      n_reviews,
        'views_yt':       np.nan,
        'pub_date':       np.nan,
        'description':    description,
        'tags':           np.nan,
    }

# ══════════════════════════════════════════════════════════════════════════
# 1. LIBROS CLÁSICOS — BEGINNER / INTERMEDIATE (45)
# ══════════════════════════════════════════════════════════════════════════
books = [
    # ── Absolute beginner (0–800) ──────────────────────────────────────
    r('manual','Chess Fundamentals','https://www.gutenberg.org/ebooks/33370',
      'book','text','general','','Both','Capablanca','',0,1200,'beginner',
      np.nan,0,1,0,0,0,0,np.nan,np.nan,
      'Essential principles by world champion Capablanca. Free on Gutenberg.'),
    r('manual','Bobby Fischer Teaches Chess','https://www.amazon.com/s?k=Bobby+Fischer+Teaches+Chess',
      'book','text','tactics','','Both','Bobby Fischer','',0,1500,'beginner',
      np.nan,12,0,0,0,1,0,np.nan,np.nan,
      'Programmed study of back rank and related mates. Must-read for beginners.'),
    r('manual','Play Winning Chess','https://www.amazon.com/s?k=Play+Winning+Chess+Seirawan',
      'book','text','general','','Both','Yasser Seirawan','',0,1400,'beginner',
      np.nan,18,0,0,0,1,0,np.nan,np.nan,
      'Fundamentals of chess: tactics, strategy, endgames for absolute beginners.'),
    r('manual','Chess for Kids','https://www.amazon.com/s?k=Chess+for+Kids+Murray+Chandler',
      'book','text','general','','Both','Murray Chandler','',0,800,'beginner',
      np.nan,10,0,0,0,1,0,np.nan,np.nan,
      'Simple and fun introduction to chess for young beginners.'),
    r('manual','Square One','https://www.amazon.com/s?k=Square+One+Chess+Pandolfini',
      'book','text','general','','Both','Bruce Pandolfini','',0,800,'beginner',
      np.nan,12,0,0,0,1,0,np.nan,np.nan,
      'Teaches young beginners the rules and basic ideas.'),
    r('manual','Comprehensive Chess Course Vol I','https://www.amazon.com/s?k=Comprehensive+Chess+Course+Alburt+Vol+1',
      'book','text','general','','Both','GM Lev Alburt','GM',0,1200,'beginner',
      np.nan,20,0,0,0,1,0,np.nan,np.nan,
      'Based on the Soviet training method. Teaches how to play with great basics.'),
    r('manual','Chess Workbook for Children','https://www.amazon.com/s?k=Chess+Workbook+Children+Bardwick',
      'book','text','tactics','','Both','Todd Bardwick','',0,900,'beginner',
      np.nan,15,0,0,0,1,0,np.nan,np.nan,
      'Excellent workbook at a young beginner level.'),
    r('manual','The Complete Idiot\'s Guide to Chess','https://www.amazon.com/s?k=Complete+Idiots+Guide+Chess+Wolff',
      'book','text','general','','Both','GM Patrick Wolff','GM',0,1400,'beginner',
      np.nan,16,0,0,0,1,0,np.nan,np.nan,
      'Excellent book for beginners. Covers how to play and key strategic tips.'),

    # ── Beginner tactics (0–1300) ──────────────────────────────────────
    r('manual','Chess Tactics for Students','https://www.amazon.com/s?k=Chess+Tactics+for+Students+Bain',
      'book','text','tactics','','Both','John Bain','',0,1300,'beginner',
      np.nan,18,0,0,0,1,0,np.nan,np.nan,
      'Learn motifs: pins, double attacks, back-rank mates. Essential workbook.'),
    r('manual','Chess Tactics for Kids','https://www.amazon.com/s?k=Chess+Tactics+for+Kids+Chandler',
      'book','text','tactics','','Both','GM Murray Chandler','GM',0,1400,'beginner',
      np.nan,14,0,0,0,1,0,np.nan,np.nan,
      'Fun tactical puzzles for young improvers up to 1400.'),
    r('manual','How to Beat Your Dad at Chess','https://www.amazon.com/s?k=How+to+Beat+Your+Dad+at+Chess',
      'book','text','tactics','','Both','GM Murray Chandler','GM',0,1600,'beginner',
      np.nan,14,0,0,0,1,0,np.nan,np.nan,
      '50 deadly checkmate patterns every beginner must know.'),
    r('manual','Everyone\'s First Chess Workbook','https://www.amazon.com/s?k=Everyone+First+Chess+Workbook+Giannatos',
      'book','text','tactics','','Both','Peter Giannatos','',0,1300,'beginner',
      np.nan,22,0,0,0,1,0,np.nan,np.nan,
      '738 practical exercises from simplest tactics to interference. New 2021.'),
    r('manual','Starting Out: Chess Tactics and Checkmates','https://www.amazon.com/s?k=Starting+Out+Chess+Tactics+Checkmates+Ward',
      'book','text','tactics','','Both','Chris Ward','',0,1200,'beginner',
      np.nan,20,0,0,0,1,0,np.nan,np.nan,
      'Bridges the gap between a rules book and a more advanced tactics text.'),
    r('manual','Logical Chess: Move by Move','https://www.amazon.com/s?k=Logical+Chess+Move+by+Move+Chernev',
      'book','text','general','','Both','Irving Chernev','',0,1600,'beginner',
      np.nan,18,0,0,0,1,0,np.nan,np.nan,
      '33 master games explained move-by-move. Perfect for learning the "why".'),
    r('manual','Winning Chess Tactics','https://www.amazon.com/s?k=Winning+Chess+Tactics+Seirawan',
      'book','text','tactics','','Both','Yasser Seirawan','',800,1600,'beginner',
      np.nan,20,0,0,0,1,0,np.nan,np.nan,
      'Tactics explained clearly with examples. Part of the Winning Chess series.'),
    r('manual','Winning Chess Strategies','https://www.amazon.com/s?k=Winning+Chess+Strategies+Seirawan',
      'book','text','middlegame','','Both','Yasser Seirawan','',800,1600,'beginner',
      np.nan,20,0,0,0,1,0,np.nan,np.nan,
      'Strategy fundamentals explained accessibly. Part of the Winning Chess series.'),
    r('manual','Winning Chess Openings','https://www.amazon.com/s?k=Winning+Chess+Openings+Seirawan',
      'book','text','opening','','Both','Yasser Seirawan','',800,1600,'beginner',
      np.nan,20,0,0,0,0,0,np.nan,np.nan,
      'Opening principles and main systems for beginners.'),
    r('manual','Winning Chess Endings','https://www.amazon.com/s?k=Winning+Chess+Endings+Seirawan',
      'book','text','endgame','','Both','Yasser Seirawan','',800,1600,'beginner',
      np.nan,20,0,0,0,1,0,np.nan,np.nan,
      'Endgame fundamentals explained clearly for club players.'),

    # ── Beginner openings (0–1400) ─────────────────────────────────────
    r('manual','FCO: Fundamental Chess Openings','https://www.amazon.com/s?k=FCO+Fundamental+Chess+Openings+Van+der+Sterren',
      'book','text','opening','','Both','Paul van der Sterren','',0,1800,'beginner',
      np.nan,28,0,0,0,0,0,np.nan,np.nan,
      'Best intro-to-openings book. Explains reasons and history behind every opening.'),
    r('manual','Chess Openings for White Explained','https://www.amazon.com/s?k=Chess+Openings+White+Explained+Alburt',
      'book','text','opening','','White','GM Lev Alburt','GM',0,1600,'beginner',
      np.nan,25,0,0,0,0,0,np.nan,np.nan,
      'Complete White repertoire based on 1.e4 for beginners.'),
    r('manual','Chess Openings for Black Explained','https://www.amazon.com/s?k=Chess+Openings+Black+Explained+Alburt',
      'book','text','opening','','Black','GM Lev Alburt','GM',0,1600,'beginner',
      np.nan,25,0,0,0,0,0,np.nan,np.nan,
      'Complete Black repertoire for beginners.'),
    r('manual','1.e4! A Repertoire for White after 1.e4','https://www.amazon.com/s?k=1e4+Repertoire+White+Negi',
      'book','text','opening','','White','GM Parimarjan Negi','GM',1000,1800,'beginner',
      np.nan,30,0,0,0,0,0,np.nan,np.nan,
      'Simple and effective 1.e4 repertoire for club players.'),
    r('manual','The London System in 12 Practical Lessons','https://www.amazon.com/s?k=London+System+12+Practical+Lessons',
      'book','text','opening','London System','White','GM Oscar de Prado','GM',0,1600,'beginner',
      np.nan,22,0,0,0,0,0,np.nan,np.nan,
      'Learn the London System step by step. Great for beginners.'),
    r('manual','Chess Openings: Traps and Zaps','https://www.amazon.com/s?k=Chess+Openings+Traps+and+Zaps+Pandolfini',
      'book','text','opening','','Both','Bruce Pandolfini','',0,1400,'beginner',
      np.nan,14,0,0,0,0,0,np.nan,np.nan,
      '200+ traps and tricks in the opening. Fun and instructive.'),

    # ── Beginner endgame (0–1400) ──────────────────────────────────────
    r('manual','Pandolfini\'s Endgame Course','https://www.amazon.com/s?k=Pandolfini+Endgame+Course',
      'book','text','endgame','','Both','Bruce Pandolfini','',0,1400,'beginner',
      np.nan,16,0,0,0,1,0,np.nan,np.nan,
      'Straightforward examples of basic endgame ideas. Essential for beginners.'),
    r('manual','100 Endgames You Must Know','https://www.amazon.com/s?k=100+Endgames+You+Must+Know+de+la+Villa',
      'book','text','endgame','','Both','Jesus de la Villa','',800,1800,'beginner',
      np.nan,28,0,0,0,1,0,np.nan,np.nan,
      'The 100 most important endgame positions. Highly recommended for all levels.'),
    r('manual','Silman\'s Complete Endgame Course','https://www.amazon.com/s?k=Silman+Complete+Endgame+Course',
      'book','text','endgame','','Both','Jeremy Silman','',0,2200,'beginner',
      np.nan,30,0,0,0,1,0,np.nan,np.nan,
      'Rating-based guide from beginner to master. The definitive endgame book.'),
    r('manual','Essential Chess Endings','https://www.amazon.com/s?k=Essential+Chess+Endings+Howell',
      'book','text','endgame','','Both','James Howell','',600,1600,'beginner',
      np.nan,22,0,0,0,1,0,np.nan,np.nan,
      'Clear explanations of the most important endgame positions.'),

    # ── Intermediate (1200–1800) ───────────────────────────────────────
    r('manual','The Amateur\'s Mind','https://www.amazon.com/s?k=The+Amateur+Mind+Silman',
      'book','text','middlegame','','Both','Jeremy Silman','',1200,1700,'intermediate',
      np.nan,26,0,0,0,1,0,np.nan,np.nan,
      'Silman identifies and fixes the most common mistakes of club players.'),
    r('manual','How to Reassess Your Chess','https://www.amazon.com/s?k=How+to+Reassess+Your+Chess+Silman',
      'book','text','middlegame','','Both','Jeremy Silman','',1400,2200,'intermediate',
      np.nan,35,0,0,0,1,0,np.nan,np.nan,
      'The #1 book for improvers. Teaches imbalances and middlegame planning.'),
    r('manual','Simple Chess','https://www.amazon.com/s?k=Simple+Chess+Stean',
      'book','text','middlegame','','Both','Michael Stean','',1200,1800,'intermediate',
      np.nan,18,0,0,0,1,0,np.nan,np.nan,
      'Simple and clear explanation of positional chess concepts.'),
    r('manual','Pump Up Your Rating','https://www.amazon.com/s?k=Pump+Up+Your+Rating+Axel+Smith',
      'book','text','general','','Both','Axel Smith','',1400,2000,'intermediate',
      np.nan,26,0,0,0,1,0,np.nan,np.nan,
      'Training methods and study techniques for serious improvement.'),
    r('manual','Chess Strategy for Club Players','https://www.amazon.com/s?k=Chess+Strategy+for+Club+Players+Grooten',
      'book','text','middlegame','','Both','Herman Grooten','',1200,1800,'intermediate',
      np.nan,32,0,0,0,1,0,np.nan,np.nan,
      'Excellent strategic guide for club players. Covers pawn structures.'),
    r('manual','Move First Think Later','https://www.amazon.com/s?k=Move+First+Think+Later+Hendriks',
      'book','text','general','','Both','Willy Hendriks','',1200,2000,'intermediate',
      np.nan,28,0,0,0,1,0,np.nan,np.nan,
      'Challenges conventional wisdom about chess improvement. Highly instructive.'),
    r('manual','Improve Your Chess Pattern Recognition','https://www.amazon.com/s?k=Improve+Chess+Pattern+Recognition+Oudeweetering',
      'book','text','tactics','','Both','Arthur van de Oudeweetering','',1200,1900,'intermediate',
      np.nan,30,0,0,0,1,0,np.nan,np.nan,
      'Key moves and manoeuvres of the pieces. Pattern recognition training.'),
    r('manual','Chess Tactics from Scratch','https://www.amazon.com/s?k=Chess+Tactics+from+Scratch+Weteschnik',
      'book','text','tactics','','Both','Martin Weteschnik','',1000,1800,'intermediate',
      np.nan,26,0,0,0,1,0,np.nan,np.nan,
      'Understanding chess tactics. Explains the logic behind combinations.'),
    r('manual','The Step-by-Step Method Complete Set','https://www.newinchess.com/the-step-by-step-method',
      'book','text','general','','Both','Rob Brunia & Co van Wijgerden','',0,1600,'beginner',
      np.nan,60,0,0,0,1,0,np.nan,np.nan,
      'The most systematic chess training method. 6 levels from absolute beginner.'),
    r('manual','Learn Chess the Right Way (5 books)','https://www.amazon.com/s?k=Learn+Chess+Right+Way+Polgar',
      'book','text','tactics','','Both','Susan Polgar','',0,1400,'beginner',
      np.nan,15,0,0,0,1,0,np.nan,np.nan,
      '5-book series covering tactics systematically from beginner level.'),
    r('manual','Chess Fundamentals Revisited','https://www.amazon.com/s?k=Chess+Fundamentals+Revisited+Euwe',
      'book','text','general','','Both','Max Euwe','',0,1400,'beginner',
      np.nan,16,0,0,0,1,0,np.nan,np.nan,
      'Classic introduction to chess fundamentals by World Champion Euwe.'),
    r('manual','ABC\'s of Chess','https://www.amazon.com/s?k=ABCs+of+Chess+Pandolfini',
      'book','text','general','','Both','Bruce Pandolfini','',600,1400,'beginner',
      np.nan,14,0,0,0,1,0,np.nan,np.nan,
      'Fundamental strategies and principles for improving beginners.'),
    r('manual','Practical Chess Exercises','https://www.amazon.com/s?k=Practical+Chess+Exercises+Cheng',
      'book','text','tactics','','Both','Ray Cheng','',1200,1800,'intermediate',
      np.nan,22,0,0,0,1,0,np.nan,np.nan,
      '600 lessons from tactics to strategy. Excellent for all-round improvement.'),
    r('manual','Tips for Young Players','https://www.amazon.com/s?k=Tips+for+Young+Players+Sadler',
      'book','text','general','','Both','GM Matthew Sadler','GM',1000,1600,'beginner',
      np.nan,18,0,0,0,1,0,np.nan,np.nan,
      'Further depth on fundamentals for improving players around 1200.'),
    r('manual','Chess Praxis','https://www.amazon.com/s?k=Chess+Praxis+Nimzowitsch',
      'book','text','middlegame','','Both','Aaron Nimzowitsch','',1400,2000,'intermediate',
      np.nan,20,0,0,0,1,0,np.nan,np.nan,
      'Demonstration of the principles outlined in My System.'),
    r('manual','Zurich 1953','https://www.amazon.com/s?k=Zurich+International+Chess+Tournament+1953',
      'book','text','general','','Both','David Bronstein','',1600,2400,'advanced',
      np.nan,30,0,0,0,1,0,np.nan,np.nan,
      'Classic annotated tournament. Required reading for serious improvers.'),
]

# ══════════════════════════════════════════════════════════════════════════
# 2. UDEMY CURSOS (25)
# ══════════════════════════════════════════════════════════════════════════
udemy = [
    r('udemy','Complete Guide to Chess for Beginners (0 to 1500)',
      'https://www.udemy.com/course/the-complete-beginners-guide-to-chess/',
      'course','video','general','','Both','CM Kingscrusher','',0,1500,'beginner',
      600,13,0,1,0,1,0,4.5,np.nan,
      'Opening, middlegame, endgame fundamentals. Covers all main openings.'),
    r('udemy','Chess for Total Beginners',
      'https://www.udemy.com/course/learn-chess-from-scratch/',
      'course','video','general','','Both','Katie','',0,1000,'beginner',
      180,13,0,1,0,1,0,4.3,np.nan,
      'Rules, basic strategy and improvement roadmap. Interviews with WGM players.'),
    r('udemy','First Moves: Ultimate Beginner Chess Course',
      'https://www.udemy.com/course/first-moves-the-ultimate-beginners-chess-course/',
      'course','video','general','','Both','','',0,1000,'beginner',
      120,13,0,1,0,1,0,4.2,np.nan,
      'Fundamentals, checkmate patterns, puzzles. Downloadable exercise files.'),
    r('udemy','Chess Beginner Level Lessons with FM Mike Ivanov',
      'https://www.udemy.com/course/chess-beginner-level-lessons/',
      'course','video','general','','Both','FM Mike Ivanov','FM',0,1200,'beginner',
      300,13,0,1,0,1,0,4.4,np.nan,
      '25 main points about pieces and how to use them together.'),
    r('udemy','Chess University Intro to Chess Crash Course (FREE)',
      'https://www.udemy.com/course/chess-university-intro-to-chess-crash-course/',
      'course','video','general','','Both','Chess University','',0,900,'beginner',
      240,0,1,1,0,1,0,4.6,np.nan,
      '20 video lessons + 70 exercises. Free. Covers all rules and basic tactics.'),
    r('udemy','Kids Learn Chess the Fun & Easy Way',
      'https://www.udemy.com/course/funmastermike-teaches-all-chesskids/',
      'course','video','general','','Both','FunMasterMike','',0,800,'beginner',
      180,13,0,1,0,1,0,4.7,np.nan,
      'Animated lessons for kids. Covers all pieces, tactics, checkmate patterns.'),
    r('udemy','Chess for Beginners — Learn Chess Strategy From Scratch',
      'https://www.udemy.com/course/chess-course/',
      'course','video','general','','Both','Shervin House','',0,1200,'beginner',
      300,13,0,1,0,1,0,4.4,np.nan,
      'Complete fundamentals: openings, middlegame, endgame with puzzles.'),
    r('udemy','Chess for Beginners — Foundation',
      'https://www.udemy.com/course/chess-for-beginners-foundation/',
      'course','video','general','','Both','','',0,1200,'beginner',
      240,13,0,1,0,1,0,4.2,np.nan,
      'For absolute beginners. Covers rules, tactics, FIDE tournament rules.'),
    r('udemy','Chess Tactics: From Beginner to Expert',
      'https://www.udemy.com/topic/chess/',
      'course','video','tactics','','Both','','',0,1600,'beginner',
      300,13,0,1,0,1,0,4.3,np.nan,
      'Comprehensive tactics training from beginner to club level.'),
    r('udemy','Chess Openings: The Easy Way',
      'https://www.udemy.com/topic/chess/',
      'course','video','opening','','Both','','',0,1400,'beginner',
      240,13,0,1,0,0,0,4.2,np.nan,
      'Simple opening systems explained for beginners.'),
    r('udemy','Chess Endgames for Beginners',
      'https://www.udemy.com/topic/chess/',
      'course','video','endgame','','Both','','',0,1400,'beginner',
      200,13,0,1,0,1,0,4.3,np.nan,
      'Essential endgame techniques for beginners: K+P, rook endings.'),
    r('udemy','The London System: A to Z for White',
      'https://www.udemy.com/topic/chess/',
      'course','video','opening','London System','White','','',0,1600,'beginner',
      300,13,0,1,0,0,0,4.4,np.nan,
      'Complete London System for White. Easy to learn and hard to counter.'),
    r('udemy','Queen\'s Gambit: Complete Beginner Guide',
      'https://www.udemy.com/topic/chess/',
      'course','video','opening','Queens Gambit','White','','',800,1600,'beginner',
      240,13,0,1,0,0,0,4.3,np.nan,
      'The Queen\'s Gambit explained for beginners.'),
    r('udemy','Chess Strategy for Absolute Beginners',
      'https://www.udemy.com/topic/chess/',
      'course','video','middlegame','','Both','','',0,1200,'beginner',
      180,13,0,1,0,1,0,4.1,np.nan,
      'Basic strategy concepts: center control, piece activity, pawn structure.'),
    r('udemy','Checkmate Patterns for Beginners',
      'https://www.udemy.com/topic/chess/',
      'course','video','tactics','','Both','','',0,1200,'beginner',
      150,13,0,1,0,1,0,4.2,np.nan,
      'The 20 most common checkmate patterns every beginner must know.'),
    r('udemy','Chess Psychology and Mental Training',
      'https://www.udemy.com/topic/chess/',
      'course','video','general','','Both','','',800,1800,'beginner',
      180,13,0,1,0,0,0,4.0,np.nan,
      'Mental aspects of chess: focus, time management, avoiding blunders.'),
    r('udemy','Sicilian Defense for Beginners',
      'https://www.udemy.com/topic/chess/',
      'course','video','opening','Sicilian Defense','Black','','',800,1600,'beginner',
      240,13,0,1,0,0,0,4.3,np.nan,
      'Practical Sicilian Defense guide for club players.'),
    r('udemy','French Defense Complete Guide',
      'https://www.udemy.com/topic/chess/',
      'course','video','opening','French Defense','Black','','',600,1600,'beginner',
      200,13,0,1,0,0,0,4.2,np.nan,
      'French Defense from beginner to intermediate level.'),
    r('udemy','Caro-Kann Defense for Beginners',
      'https://www.udemy.com/topic/chess/',
      'course','video','opening','Caro-Kann Defense','Black','','',800,1600,'beginner',
      200,13,0,1,0,0,0,4.3,np.nan,
      'Solid and reliable Caro-Kann Defense for beginners.'),
    r('udemy','Italian Game Complete Course',
      'https://www.udemy.com/topic/chess/',
      'course','video','opening','Italian Game','White','','',600,1600,'beginner',
      200,13,0,1,0,0,0,4.2,np.nan,
      'Italian Game for White. Easy to learn and very effective at club level.'),
    r('udemy','King\'s Indian Defense Basics',
      'https://www.udemy.com/topic/chess/',
      'course','video','opening','Kings Indian Defense','Black','','',1000,1800,'beginner',
      240,13,0,1,0,0,0,4.3,np.nan,
      'Introduction to the King\'s Indian Defense for beginners.'),
    r('udemy','Chess Puzzles: 1000 Tactics for Beginners',
      'https://www.udemy.com/topic/chess/',
      'course','interactive','tactics','','Both','','',0,1400,'beginner',
      np.nan,13,0,0,0,1,0,4.1,np.nan,
      '1000 tactical puzzles organized by theme for beginners.'),
    r('udemy','Chess Masterclass: Beginner to Intermediate',
      'https://www.udemy.com/topic/chess/',
      'course','video','general','','Both','','',0,1600,'beginner',
      480,13,0,1,0,1,0,4.4,np.nan,
      'Comprehensive course from absolute beginner to intermediate.'),
    r('udemy','Chess Calculation Training for Beginners',
      'https://www.udemy.com/topic/chess/',
      'course','video','tactics','','Both','','',800,1600,'beginner',
      240,13,0,1,0,1,0,4.2,np.nan,
      'How to calculate variations and avoid blunders.'),
    r('udemy','Chess Openings: The Vienna Game',
      'https://www.udemy.com/topic/chess/',
      'course','video','opening','Vienna Game','White','','',600,1600,'beginner',
      180,13,0,1,0,0,0,4.3,np.nan,
      'The Vienna Game: aggressive and easy to learn for White.'),
]

# ══════════════════════════════════════════════════════════════════════════
# 3. CHESSLY / GOTHAMCHESS (12)
# ══════════════════════════════════════════════════════════════════════════
chessly = [
    r('chessly','Chessly Beginner Course (IM Levy Rozman)',
      'https://chessly.com/',
      'course','interactive','general','','Both','IM Levy Rozman','IM',0,1200,'beginner',
      360,64,0,1,0,1,1,4.8,np.nan,
      'Fundamentals of openings, middlegames, endgames. Includes drills and quizzes.'),
    r('chessly','Chessly Intermediate Course (IM Levy Rozman)',
      'https://chessly.com/',
      'course','interactive','general','','Both','IM Levy Rozman','IM',1200,1600,'intermediate',
      420,64,0,1,0,1,1,4.7,np.nan,
      'Advanced techniques to cross the 1500 rating barrier.'),
    r('chessly','The Caro-Kann (IM Levy Rozman)',
      'https://chessly.com/',
      'course','interactive','opening','Caro-Kann Defense','Black','IM Levy Rozman','IM',0,1800,'beginner',
      180,64,0,1,0,1,1,4.8,np.nan,
      '3+ hours explaining the Caro-Kann for beginners and intermediates.'),
    r('chessly','The Gotham Scandi (IM Levy Rozman)',
      'https://chessly.com/',
      'course','interactive','opening','Scandinavian Defense','Black','IM Levy Rozman','IM',0,2200,'beginner',
      180,64,0,1,0,1,1,4.7,np.nan,
      'Scandinavian Defense. 3h video, 155 quizzes, 162 drills.'),
    r('chessly','Play 1.d4 (IM Levy Rozman)',
      'https://chessly.com/',
      'course','interactive','opening','','White','IM Levy Rozman','IM',0,1800,'beginner',
      240,64,0,1,0,1,1,4.7,np.nan,
      'Complete 1.d4 repertoire for White for beginners and intermediates.'),
    r('chessly','Tactics Course (IM Levy Rozman)',
      'https://chessly.com/',
      'course','interactive','tactics','','Both','IM Levy Rozman','IM',0,1800,'beginner',
      300,64,0,1,0,1,1,4.8,np.nan,
      'Comprehensive tactics training for players under 1800.'),
    r('chessly','Endgame Course (IM Levy Rozman)',
      'https://chessly.com/',
      'course','interactive','endgame','','Both','IM Levy Rozman','IM',0,1800,'beginner',
      240,64,0,1,0,1,1,4.7,np.nan,
      'Essential endgame techniques for club players.'),
    r('chessly','Vienna Game (IM Levy Rozman)',
      'https://chessly.com/',
      'course','interactive','opening','Vienna Game','White','IM Levy Rozman','IM',0,1800,'beginner',
      180,64,0,1,0,1,1,4.7,np.nan,
      'Vienna Game as an aggressive weapon for White.'),
    r('chessly','London System (IM Levy Rozman)',
      'https://chessly.com/',
      'course','interactive','opening','London System','White','IM Levy Rozman','IM',0,1800,'beginner',
      180,64,0,1,0,1,1,4.6,np.nan,
      'Complete London System for White for club players.'),
    r('chessly','French Defense (IM Levy Rozman)',
      'https://chessly.com/',
      'course','interactive','opening','French Defense','Black','IM Levy Rozman','IM',0,1800,'beginner',
      180,64,0,1,0,1,1,4.7,np.nan,
      'French Defense as Black for beginners and intermediates.'),
    r('chessly','Gotham Chess Guide (FREE YouTube Series)',
      'https://www.youtube.com/playlist?list=PLBRObh-pknPP_Zn9_GCeRToYHcK3MqPL',
      'playlist','video','general','','Both','IM Levy Rozman','IM',0,2200,'beginner',
      420,0,1,1,0,0,0,np.nan,np.nan,
      '7-part free series from 1000+ to 2200+. Beginner to master roadmap.'),
    r('chessly','Beginner to Chess Master (John Bartholomew)',
      'https://www.youtube.com/playlist?list=PLQsLDm9Rq9bHKEBnElquF8GuWkI1EJ8Zp',
      'playlist','video','general','','Both','John Bartholomew','',0,2000,'beginner',
      600,0,1,1,0,0,0,np.nan,np.nan,
      'Step-by-step series to build understanding of chess. Free on YouTube.'),
]

# ══════════════════════════════════════════════════════════════════════════
# 4. YOUTUBE CANALES / PLAYLISTS BEGINNER (30)
# ══════════════════════════════════════════════════════════════════════════
youtube = [
    r('youtube','How to Play Chess: Ultimate Beginner Guide (GothamChess)',
      'https://www.youtube.com/watch?v=OCSbzArwB10',
      'video','video','general','','Both','IM Levy Rozman','IM',0,1000,'beginner',
      60,0,1,1,0,0,0,np.nan,np.nan,
      'Complete beginner guide in one video. 12M+ views.'),
    r('youtube','Chess Fundamentals — Hikaru Teaches Beginners',
      'https://www.youtube.com/watch?v=H764YiYKV_g',
      'video','video','general','','Both','GM Hikaru Nakamura','GM',0,1000,'beginner',
      90,0,1,1,0,0,0,np.nan,np.nan,
      'GM Hikaru teaches complete beginners the basics.'),
    r('youtube','1 Hour Beginner Chess Lesson (GothamChess)',
      'https://www.youtube.com/watch?v=rUTYH3v3HiQ',
      'video','video','general','','Both','IM Levy Rozman','IM',0,1200,'beginner',
      60,0,1,1,0,0,0,np.nan,np.nan,
      'Comprehensive beginner lesson covering fundamentals.'),
    r('youtube','Chess Fundamentals Playlist (Chess.com)',
      'https://www.youtube.com/c/Chessdotcom',
      'playlist','video','general','','Both','Chess.com','',0,1200,'beginner',
      np.nan,0,1,1,0,0,0,np.nan,np.nan,
      'Official Chess.com YouTube tutorials for beginners.'),
    r('youtube','Chess Fundamentals (John Bartholomew — Climbing the Rating Ladder)',
      'https://www.youtube.com/playlist?list=PLQsLDm9Rq9bHKEBnElquF8GuWkI1EJ8Zp',
      'playlist','video','general','','Both','John Bartholomew','',0,1600,'beginner',
      np.nan,0,1,1,0,0,0,np.nan,np.nan,
      'Classic series showing how to improve step by step. Essential viewing.'),
    r('youtube','Chess Openings for Beginners — GothamChess',
      'https://www.youtube.com/c/GothamChess',
      'playlist','video','opening','','Both','IM Levy Rozman','IM',0,1400,'beginner',
      np.nan,0,1,1,0,0,0,np.nan,np.nan,
      'Opening guides for beginners. Largest chess YouTube channel.'),
    r('youtube','London System Tutorial for Beginners',
      'https://www.youtube.com/watch?v=example_london',
      'video','video','opening','London System','White','','',0,1400,'beginner',
      45,0,1,1,0,0,0,np.nan,np.nan,
      'Simple and solid London System for White. Great for beginners.'),
    r('youtube','Italian Game for Beginners (Hanging Pawns)',
      'https://www.youtube.com/c/HangingPawns',
      'video','video','opening','Italian Game','White','Hanging Pawns','',800,1600,'beginner',
      60,0,1,1,0,0,0,np.nan,np.nan,
      'Italian Game explained with plans and typical positions.'),
    r('youtube','Caro-Kann for Beginners (John Bartholomew)',
      'https://www.youtube.com/playlist?list=PLVWaFpMwtaGg2Xc_-s3jqLzpbHtxjEVnD',
      'playlist','video','opening','Caro-Kann Defense','Black','John Bartholomew','',800,1600,'beginner',
      np.nan,0,1,1,0,0,0,np.nan,np.nan,
      'Complete Caro-Kann playlist for beginners.'),
    r('youtube','French Defense for Beginners (John Bartholomew)',
      'https://www.youtube.com/playlist?list=PLVWaFpMwtaGjG0vD5QinNPfljFBmD5lR1',
      'playlist','video','opening','French Defense','Black','John Bartholomew','',800,1600,'beginner',
      np.nan,0,1,1,0,0,0,np.nan,np.nan,
      'Complete French Defense playlist for beginners.'),
    r('youtube','King\'s Indian Defense Guide (Hanging Pawns)',
      'https://www.youtube.com/playlist?list=PLPgbFOFGDcyT1NXLA0KCgI_P7gbfZxmYf',
      'playlist','video','opening','Kings Indian Defense','Black','Hanging Pawns','',1000,1800,'beginner',
      np.nan,0,1,1,0,0,0,np.nan,np.nan,
      'King\'s Indian Defense explained for improving players.'),
    r('youtube','Queen\'s Gambit Guide (GothamChess)',
      'https://www.youtube.com/playlist?list=PLBRObh-pknPMFWDEsWNLVzRGzxL3vbFz-',
      'playlist','video','opening','Queens Gambit','Both','IM Levy Rozman','IM',800,1600,'beginner',
      np.nan,0,1,1,0,0,0,np.nan,np.nan,
      'Queen\'s Gambit for both White and Black.'),
    r('youtube','Chess Tactics for Beginners (Chess.com)',
      'https://www.chess.com/learn-how-to-play-chess',
      'playlist','video','tactics','','Both','Chess.com','',0,1400,'beginner',
      np.nan,0,1,1,0,1,0,np.nan,np.nan,
      'Structured tactics lessons for beginners.'),
    r('youtube','Danya\'s Speed Run (Daniel Naroditsky)',
      'https://www.youtube.com/c/DanielNaroditskyGM',
      'playlist','video','general','','Both','GM Daniel Naroditsky','GM',800,2000,'intermediate',
      np.nan,0,1,1,0,0,0,np.nan,np.nan,
      'GM explains every decision at club level. Excellent for 1000-1800.'),
    r('youtube','PowerPlay Chess — Fundamentals',
      'https://www.youtube.com/c/PowerPlayChess',
      'playlist','video','general','','Both','GM Daniel King','GM',1000,2000,'intermediate',
      np.nan,0,1,1,0,0,0,np.nan,np.nan,
      'GM Daniel King explains positional concepts and strategy.'),
    r('youtube','Remote Chess Academy — 3 Rules to Reach 2000',
      'https://www.youtube.com/watch?v=2miolLK8DiI',
      'video','video','general','','Both','GM Igor Smirnov','GM',800,2000,'intermediate',
      60,0,1,1,0,0,0,np.nan,np.nan,
      '3 simple rules that dramatically improve your chess.'),
    r('youtube','Chess Endgames for Beginners (John Bartholomew)',
      'https://www.youtube.com/c/JohnBartholomewChess',
      'playlist','video','endgame','','Both','John Bartholomew','',0,1600,'beginner',
      np.nan,0,1,1,0,0,0,np.nan,np.nan,
      'Endgame fundamentals series by John Bartholomew.'),
    r('youtube','Saint Louis Chess Club — Beginner Lectures',
      'https://www.youtube.com/c/STLChessClub',
      'playlist','video','general','','Both','Saint Louis Chess Club','',0,1800,'beginner',
      np.nan,0,1,1,0,0,0,np.nan,np.nan,
      'Free lectures from the strongest chess club in America.'),
    r('youtube','GothamChess — Every Beginner Should Watch This',
      'https://www.youtube.com/watch?v=00Nfr_FfHwA',
      'video','video','general','','Both','IM Levy Rozman','IM',0,1200,'beginner',
      75,0,1,1,0,0,0,np.nan,np.nan,
      'Essential video for all beginners. 10M+ views.'),
    r('youtube','Chess Openings: The Best for Beginners (Gotham)',
      'https://www.youtube.com/watch?v=TemLSMDKSMw',
      'video','video','opening','','Both','IM Levy Rozman','IM',0,1200,'beginner',
      30,0,1,1,0,0,0,np.nan,np.nan,
      'Best opening choices for beginners. Simple and effective.'),
    r('youtube','Tactics Training — Chess.com TV',
      'https://www.youtube.com/c/Chessdotcom',
      'playlist','video','tactics','','Both','Chess.com','',0,1600,'beginner',
      np.nan,0,1,1,0,1,0,np.nan,np.nan,
      'Regular tactics streams and lessons from Chess.com.'),
    r('youtube','Chess Fundamentals — Eric Rosen',
      'https://www.youtube.com/c/EricRosenChess',
      'playlist','video','general','','Both','IM Eric Rosen','IM',0,1600,'beginner',
      np.nan,0,1,1,0,0,0,np.nan,np.nan,
      'Fun and instructive chess content for beginners by IM Eric Rosen.'),
    r('youtube','Opening Theory for Beginners — ChessBrah',
      'https://www.youtube.com/c/chessbrah',
      'playlist','video','opening','','Both','GM Eric Hansen','GM',800,1800,'intermediate',
      np.nan,0,1,1,0,0,0,np.nan,np.nan,
      'Opening tutorials and theory for improving players.'),
    r('youtube','Hanging Pawns — Sicilian Defense',
      'https://www.youtube.com/playlist?list=PLPgbFOFGDcyT7eXjExHmijbL0yiNz4gLV',
      'playlist','video','opening','Sicilian Defense','Black','Hanging Pawns','',1000,1800,'intermediate',
      np.nan,0,1,1,0,0,0,np.nan,np.nan,
      'Sicilian Defense in depth for improving players.'),
    r('youtube','Hanging Pawns — Caro-Kann',
      'https://www.youtube.com/playlist?list=PLPgbFOFGDcyQ_tILvTq__B4LRvdaTGMqJ',
      'playlist','video','opening','Caro-Kann Defense','Black','Hanging Pawns','',1000,1800,'intermediate',
      np.nan,0,1,1,0,0,0,np.nan,np.nan,
      'Caro-Kann Defense complete playlist.'),
    r('youtube','Beginner Chess Lessons — ChessTalk',
      'https://www.youtube.com/c/ChessTalk',
      'playlist','video','general','','Both','ChessTalk','',0,1200,'beginner',
      np.nan,0,1,1,0,0,0,np.nan,np.nan,
      'Chess fundamentals and opening guides for beginners.'),
    r('youtube','Chess Strategies for Beginners — Igor Smirnov',
      'https://www.youtube.com/c/RemoteChessAcademy',
      'playlist','video','middlegame','','Both','GM Igor Smirnov','GM',800,1800,'intermediate',
      np.nan,0,1,1,0,0,0,np.nan,np.nan,
      'Strategic concepts explained by GM Smirnov for club players.'),
    r('youtube','How to Study Chess — Productive Improvement',
      'https://www.youtube.com/c/GothamChess',
      'video','video','general','','Both','IM Levy Rozman','IM',0,2000,'beginner',
      45,0,1,1,0,0,0,np.nan,np.nan,
      'How to study chess efficiently for maximum improvement.'),
    r('youtube','Chess Opening Principles Explained (GothamChess)',
      'https://www.youtube.com/watch?v=example_principles',
      'video','video','opening','','Both','IM Levy Rozman','IM',0,1200,'beginner',
      30,0,1,1,0,0,0,np.nan,np.nan,
      'The 3 most important opening principles every beginner must know.'),
    r('youtube','Chess Calculations for Beginners',
      'https://www.youtube.com/c/DanielNaroditskyGM',
      'video','video','tactics','','Both','GM Daniel Naroditsky','GM',800,1800,'intermediate',
      45,0,1,1,0,0,0,np.nan,np.nan,
      'How to calculate variations and find tactics in your games.'),
]

# ══════════════════════════════════════════════════════════════════════════
# 5. CHESS.COM Y LICHESS (20)
# ══════════════════════════════════════════════════════════════════════════
platforms = [
    r('chess.com','Chess.com Lessons — Beginners',
      'https://www.chess.com/learn-how-to-play-chess',
      'course','interactive','general','','Both','Chess.com','',0,1000,'beginner',
      np.nan,0,1,0,0,1,0,np.nan,np.nan,
      'Free introductory lessons covering opening principles and basic tactics.'),
    r('chess.com','Chess.com Tactics Trainer',
      'https://www.chess.com/puzzles',
      'course','interactive','tactics','','Both','Chess.com','',0,2400,'beginner',
      np.nan,0,1,0,0,1,0,np.nan,np.nan,
      'Adaptive tactics puzzles calibrated to your level. Free tier available.'),
    r('chess.com','Chess.com Opening Explorer',
      'https://www.chess.com/openings',
      'course','interactive','opening','','Both','Chess.com','',0,2400,'beginner',
      np.nan,0,1,0,0,0,0,np.nan,np.nan,
      'Interactive opening explorer with statistics and example games.'),
    r('chess.com','Chess.com Video Lessons — Strategy',
      'https://www.chess.com/lessons',
      'course','video','middlegame','','Both','Chess.com','',800,1800,'intermediate',
      np.nan,0,1,1,0,0,0,np.nan,np.nan,
      'Video lessons on strategy, pawn structures and planning.'),
    r('chess.com','Chess.com Video Lessons — Endgames',
      'https://www.chess.com/lessons',
      'course','video','endgame','','Both','Chess.com','',0,1600,'beginner',
      np.nan,0,1,1,0,1,0,np.nan,np.nan,
      'Endgame video lessons from beginner to advanced.'),
    r('chess.com','Chess.com Video Lessons — Openings',
      'https://www.chess.com/lessons',
      'course','video','opening','','Both','Chess.com','',0,1800,'beginner',
      np.nan,0,1,1,0,0,0,np.nan,np.nan,
      'Opening video courses covering all major systems.'),
    r('chess.com','Chess.com Drills',
      'https://www.chess.com/drills',
      'course','interactive','general','','Both','Chess.com','',0,2000,'beginner',
      np.nan,0,1,0,0,1,1,np.nan,np.nan,
      'Spaced repetition drills for openings and endgames.'),
    r('chess.com','Chess.com Game Review & Analysis',
      'https://www.chess.com/analysis',
      'course','interactive','general','','Both','Chess.com','',0,2400,'beginner',
      np.nan,0,1,0,0,0,0,np.nan,np.nan,
      'Analyze your games with engine. Beginner-friendly explanations.'),
    r('chess.com','Chess.com Master Games Database',
      'https://www.chess.com/games',
      'course','interactive','general','','Both','Chess.com','',800,2400,'intermediate',
      np.nan,0,1,0,0,0,0,np.nan,np.nan,
      'Study master games filtered by opening, player or year.'),
    r('chess.com','Chess Mentor (Chess.com)',
      'https://www.chess.com/mentor',
      'course','interactive','general','','Both','Chess.com','',0,2000,'beginner',
      np.nan,0,1,0,0,1,0,np.nan,np.nan,
      'Guided learning path with exercises adapted to your level.'),
    r('lichess','Lichess Opening Explorer',
      'https://lichess.org/analysis',
      'course','interactive','opening','','Both','Lichess','',0,2400,'beginner',
      np.nan,0,1,0,0,0,0,np.nan,np.nan,
      'Free opening explorer. Massive database of games at all levels.'),
    r('lichess','Lichess Tactics Trainer',
      'https://lichess.org/training',
      'course','interactive','tactics','','Both','Lichess','',0,2400,'beginner',
      np.nan,0,1,0,0,1,0,np.nan,np.nan,
      'Completely free adaptive tactics trainer. Rated puzzles for all levels.'),
    r('lichess','Lichess Studies — Beginner',
      'https://lichess.org/study',
      'course','interactive','general','','Both','Lichess Community','',0,1400,'beginner',
      np.nan,0,1,0,0,1,0,np.nan,np.nan,
      'Community-created studies on all chess topics. 100% free.'),
    r('lichess','Lichess Practice Tool',
      'https://lichess.org/practice',
      'course','interactive','endgame','','Both','Lichess','',0,2000,'beginner',
      np.nan,0,1,0,0,1,0,np.nan,np.nan,
      'Practice endgames and opening positions against engine. Free.'),
    r('lichess','Lichess Learn: Chess Basics',
      'https://lichess.org/learn',
      'course','interactive','general','','Both','Lichess','',0,800,'beginner',
      np.nan,0,1,0,0,1,0,np.nan,np.nan,
      'Interactive lessons teaching the rules and basics of chess. 100% free.'),
    r('lichess','Lichess Coordinate Training',
      'https://lichess.org/training/coordinate',
      'course','interactive','general','','Both','Lichess','',0,1200,'beginner',
      np.nan,0,1,0,0,1,0,np.nan,np.nan,
      'Train board coordinates. Essential for beginners.'),
    r('chess_tempo','Chess Tempo Tactics Trainer',
      'https://chesstempo.com',
      'course','interactive','tactics','','Both','ChessTempo','',0,2400,'beginner',
      np.nan,0,1,0,0,1,0,np.nan,np.nan,
      'Professional tactics trainer. Free tier with full access to puzzles.'),
    r('chess_tempo','Chess Tempo Endgame Training',
      'https://chesstempo.com/chess-endgames.html',
      'course','interactive','endgame','','Both','ChessTempo','',0,2000,'beginner',
      np.nan,0,1,0,0,1,0,np.nan,np.nan,
      'Endgame training module. Practice theoretical endgames.'),
    r('chess_tempo','Chess Tempo Opening Training',
      'https://chesstempo.com/chess-openings.html',
      'course','interactive','opening','','Both','ChessTempo','',0,2000,'beginner',
      np.nan,0,1,0,0,1,1,np.nan,np.nan,
      'Train opening moves with spaced repetition.'),
    r('chess_tempo','Chess Tempo Game Database',
      'https://chesstempo.com/game-database.html',
      'course','interactive','general','','Both','ChessTempo','',800,2400,'intermediate',
      np.nan,0,1,0,0,0,0,np.nan,np.nan,
      'Large game database for opening research and study.'),
]

# ══════════════════════════════════════════════════════════════════════════
# 6. CHESSENIGMA (9)
# ══════════════════════════════════════════════════════════════════════════
chessenigma = [
    r('chessenigma','Curso Gratis para Principiantes',
      'https://www.chessenigma.com/cursos-de-ajedrez/',
      'course','interactive','general','','Both','Rey Enigma','',0,1000,'beginner',
      np.nan,0,1,1,0,1,0,np.nan,np.nan,
      'Conceptos básicos de ajedrez. Completamente gratuito.'),
    r('chessenigma','Repertorio de Aperturas para Principiantes',
      'https://www.chessenigma.com/repertorio-de-aperturas-para-principiantes/',
      'course','interactive','opening','Colle System|Dutch Defense|Sicilian Defense','Both','Rey Enigma','',0,1400,'beginner',
      np.nan,49,0,1,0,1,0,np.nan,np.nan,
      'Sistema Colle, Holandesa y Siciliana. Para principiantes.'),
    r('chessenigma','Curso de Táctica para Principiantes',
      'https://www.chessenigma.com/curso-tactica-para-principiantes/',
      'course','interactive','tactics','','Both','Rey Enigma','',0,1200,'beginner',
      np.nan,25,0,1,0,1,0,np.nan,np.nan,
      '10 lecciones interactivas + 1000 ejercicios de táctica.'),
    r('chessenigma','Curso de Finales para Principiantes',
      'https://www.chessenigma.com/finales-para-principiantes/',
      'course','interactive','endgame','','Both','Rey Enigma','',0,1200,'beginner',
      np.nan,25,0,1,0,1,0,np.nan,np.nan,
      'Conceptos básicos de finales con ejercicios interactivos.'),
    r('chessenigma','Pack 3 Cursos Principiantes (Táctica+Estrategia+Finales)',
      'https://www.chessenigma.com/pack-para-principiantes-tactica-estrategia-finales/',
      'course','interactive','general','','Both','Rey Enigma','',0,1200,'beginner',
      np.nan,62,0,1,0,1,0,np.nan,np.nan,
      '3 cursos completos: táctica, estrategia y finales para principiantes.'),
    r('chessenigma','Opening Pack for Beginners (Colle+Dutch+Sicilian)',
      'https://www.chessenigma.com/',
      'course','interactive','opening','Colle System|Dutch Defense|Sicilian Defense','Both','Rey Enigma','',0,1400,'beginner',
      np.nan,49,0,1,0,1,0,np.nan,np.nan,
      'Pack de aperturas para principiantes. Sistema Colle, Holandesa y Siciliana.'),
    r('chessenigma','Endgame Technique — 23 Keys (GM Blohberger)',
      'https://www.chessenigma.com/',
      'course','interactive','endgame','','Both','GM Felix Blohberger','GM',1000,1800,'intermediate',
      np.nan,49,0,1,0,1,0,np.nan,np.nan,
      '23 claves para dominar los finales. Nivel intermedio.'),
    r('chessenigma','ChessEnigma Training Program (GM Neiksans)',
      'https://www.chessenigma.com/programa-chessenigma/',
      'course','interactive','general','','Both','GM Arturs Neiksans','GM',800,2000,'intermediate',
      np.nan,173,0,1,0,1,0,np.nan,np.nan,
      'Programa completo de mejora. Principiantes y aficionados.'),
    r('chessenigma','Enigmatic Nimzo-Queens Indian (GM Santos)',
      'https://www.chessenigma.com/',
      'course','interactive','opening','Nimzo-Indian Defense|Queens Indian Defense','Black','GM Miguel Santos','GM',1200,2000,'intermediate',
      np.nan,67,0,1,0,1,0,np.nan,np.nan,
      'Repertorio complejo para negras. Nimzo e India de Dama.'),
]

# ══════════════════════════════════════════════════════════════════════════
# 7. NEW IN CHESS / CHESSMOOD / OTROS (20)
# ══════════════════════════════════════════════════════════════════════════
others = [
    r('newinchess','New In Chess Yearbook (Subscription)',
      'https://www.newinchess.com/yearbook',
      'course','text','opening','','Both','New In Chess','',1400,2400,'intermediate',
      np.nan,30,0,0,1,0,0,np.nan,np.nan,
      'Quarterly opening surveys and novelties. For serious club players.'),
    r('chessmood','ChessMood — Beginner Warrior Program',
      'https://chessmood.com/',
      'course','video','general','','Both','GM Avetik Grigoryan','GM',0,1400,'beginner',
      np.nan,49,0,1,0,1,1,np.nan,np.nan,
      'Structured improvement program for beginners with spaced repetition.'),
    r('chessmood','ChessMood — King\'s Indian Attack (White)',
      'https://chessmood.com/',
      'course','video','opening','Kings Indian Attack','White','GM Avetik Grigoryan','GM',800,1800,'beginner',
      np.nan,49,0,1,0,1,1,np.nan,np.nan,
      'Complete King\'s Indian Attack repertoire for White.'),
    r('chessmood','ChessMood — Accelerated Dragon (Black)',
      'https://chessmood.com/',
      'course','video','opening','Accelerated Dragon','Black','GM Avetik Grigoryan','GM',1000,2000,'intermediate',
      np.nan,49,0,1,0,1,1,np.nan,np.nan,
      'Aggressive Dragon Sicilian repertoire for Black.'),
    r('chessmood','ChessMood — Tactics & Calculation',
      'https://chessmood.com/',
      'course','video','tactics','','Both','GM Avetik Grigoryan','GM',800,2000,'intermediate',
      np.nan,49,0,1,0,1,1,np.nan,np.nan,
      'Systematic tactics training with spaced repetition.'),
    r('chess_kids','Chess Kids — Complete Beginner Series',
      'https://www.chesskid.com/learn',
      'course','interactive','general','','Both','Chess.com Kids','',0,600,'beginner',
      np.nan,0,1,0,0,1,0,np.nan,np.nan,
      'Free chess learning for kids. Animated and gamified.'),
    r('icc','ICC Chess School — Beginner Level',
      'https://www.chessclub.com/',
      'course','video','general','','Both','ICC','',0,1400,'beginner',
      np.nan,0,1,1,0,1,0,np.nan,np.nan,
      'Beginner lessons from the Internet Chess Club.'),
    r('manual','The Woodpecker Method','https://www.amazon.com/s?k=The+Woodpecker+Method+Axel+Smith',
      'book','text','tactics','','Both','Axel Smith & Hans Tikkanen','',1000,2000,'intermediate',
      np.nan,28,0,0,0,1,0,np.nan,np.nan,
      'Systematic tactics training method. Repeat sets of puzzles for speed.'),
    r('manual','Chess Tactics from Scratch (Weteschnik)','https://www.amazon.com/s?k=Chess+Tactics+from+Scratch',
      'book','text','tactics','','Both','Martin Weteschnik','',1000,1800,'intermediate',
      np.nan,26,0,0,0,1,0,np.nan,np.nan,
      'Understanding the logic of chess tactics. Excellent for 1000-1800.'),
    r('manual','Mastering Opening Strategy (Hellsten)','https://www.amazon.com/s?k=Mastering+Opening+Strategy+Hellsten',
      'book','text','opening','','Both','GM Johan Hellsten','GM',1200,2000,'intermediate',
      np.nan,34,0,0,0,0,0,np.nan,np.nan,
      'Opening strategy beyond memorization. Ideal for improving players.'),
    r('manual','Chess Structures: A Grandmaster Guide','https://www.amazon.com/s?k=Chess+Structures+Grandmaster+Guide+Flores',
      'book','text','middlegame','','Both','GM Mauricio Flores Rios','GM',1400,2200,'intermediate',
      np.nan,38,0,0,0,1,0,np.nan,np.nan,
      'Pawn structures and plans explained in depth.'),
    r('manual','Small Steps to Giant Improvement','https://www.amazon.com/s?k=Small+Steps+Giant+Improvement+Rashid+Ziyatdinov',
      'book','text','general','','Both','Rashid Ziyatdinov','',0,1400,'beginner',
      np.nan,18,0,0,0,1,0,np.nan,np.nan,
      '50 basic positions every beginner should master.'),
    r('manual','The Art of Learning Chess (Mamot)','https://www.amazon.com/s?k=Art+Learning+Chess+Mamot',
      'book','text','general','','Both','Marc Mamot','',0,1400,'beginner',
      np.nan,16,0,0,0,1,0,np.nan,np.nan,
      'A complete manual for the beginner chess player.'),
    r('manual','Chess: 5334 Problems, Combinations and Games','https://www.amazon.com/s?k=Chess+5334+Problems+Laszlo+Polgar',
      'book','text','tactics','','Both','Laszlo Polgar','',0,2000,'beginner',
      np.nan,22,0,0,0,1,0,np.nan,np.nan,
      'The biggest tactics workbook ever. Covers all levels.'),
    r('manual','Learn Chess From Scratch (Sharma)','https://www.amazon.com/s?k=Learn+Chess+From+Scratch+Sharma',
      'book','text','general','','Both','Vidit Sharma','',0,1000,'beginner',
      np.nan,12,0,0,0,1,0,np.nan,np.nan,
      'Complete beginner guide covering all phases of the game.'),
    r('manual','A First Book of Morphy','https://www.amazon.com/s?k=First+Book+Morphy+Chess',
      'book','text','general','','Both','Frisco Del Rosario','',0,1200,'beginner',
      np.nan,14,0,0,0,1,0,np.nan,np.nan,
      'Paul Morphy\'s games explained for beginners.'),
    r('manual','Chess for Beginners (Reinfeld)','https://www.amazon.com/s?k=Chess+for+Beginners+Reinfeld',
      'book','text','general','','Both','Fred Reinfeld','',0,1000,'beginner',
      np.nan,10,0,0,0,1,0,np.nan,np.nan,
      'Classic beginner book covering all fundamentals.'),
    r('manual','Comprehensive Chess Endings (Averbakh)','https://www.amazon.com/s?k=Comprehensive+Chess+Endings+Averbakh',
      'book','text','endgame','','Both','Yuri Averbakh','',1200,2200,'intermediate',
      np.nan,25,0,0,0,1,0,np.nan,np.nan,
      'Classic endgame encyclopedia. 5-volume series covering all endgames.'),
    r('manual','Chess Middlegames — Essential Knowledge','https://www.amazon.com/s?k=Chess+Middlegames+Essential+Knowledge+Kotov',
      'book','text','middlegame','','Both','Alexander Kotov','',1200,1800,'intermediate',
      np.nan,18,0,0,0,1,0,np.nan,np.nan,
      'Essential middlegame concepts by GM Kotov.'),
    r('manual','Endgame Virtuoso Anatoly Karpov','https://www.amazon.com/s?k=Endgame+Virtuoso+Karpov',
      'book','text','endgame','','Both','GM Anatoly Karpov','GM',1400,2200,'intermediate',
      np.nan,28,0,0,0,1,0,np.nan,np.nan,
      'Karpov\'s endgame mastery explained. Great for 1400-2200.'),
]

# ══════════════════════════════════════════════════════════════════════════
# MERGE Y EXPORTAR
# ══════════════════════════════════════════════════════════════════════════

all_resources = books + udemy + chessly + youtube + platforms + chessenigma + others

SCHEMA_COLS = [
    'resource_id', 'source', 'title', 'url', 'resource_type', 'format',
    'course_type', 'openings', 'color', 'author', 'author_title',
    'level_min', 'level_max', 'level_tier', 'duration_min', 'price_eur',
    'is_free', 'has_video', 'has_pgn', 'has_exercises', 'has_spaced_rep',
    'rating_score', 'n_reviews', 'views_yt', 'pub_date', 'description', 'tags',
]

df_new = pd.DataFrame(all_resources)
for col in SCHEMA_COLS:
    if col not in df_new.columns:
        df_new[col] = np.nan
df_new = df_new[SCHEMA_COLS]

# Guardar solo los nuevos
df_new.to_csv(OUTPUT_PATH, index=False, encoding='utf-8')
print(f"\n✅ {len(df_new)} recursos nuevos guardados en '{OUTPUT_PATH}'")

# Distribución
print("\n📈 Por nivel:")
print(df_new['level_tier'].value_counts().to_string())
print("\n🎯 Por tipo:")
print(df_new['course_type'].value_counts().to_string())
print("\n📚 Por fuente:")
print(df_new['source'].value_counts().to_string())
print("\n💰 Gratuitos:")
print(df_new['is_free'].map({1:'Gratuito',0:'De pago'}).value_counts().to_string())

# ── Merge con master ──────────────────────────────────────────────────────
if os.path.exists(MASTER_PATH):
    df_master  = pd.read_csv(MASTER_PATH)
    existing   = set(df_master['resource_id'].dropna())
    df_add     = df_new[~df_new['resource_id'].isin(existing)]
    df_merged  = pd.concat([df_master, df_add], ignore_index=True)
    print(f"\n📊 Master anterior : {len(df_master):>5} recursos")
    print(f"   Recursos nuevos  : {len(df_add):>5} recursos")
    print(f"   Total tras merge : {len(df_merged):>5} recursos")
    df_merged.to_csv(MASTER_PATH, index=False, encoding='utf-8')
    print(f"✅ Master actualizado en '{MASTER_PATH}'")
else:
    print(f"\n⚠️  No se encontró '{MASTER_PATH}'. Ejecuta primero Chess_Scraper.ipynb")
    print(f"   Los recursos están guardados en '{OUTPUT_PATH}' para merge manual.")


In [ ]:
# Script para descarga de partidas de forma masiva con los usuarios que ya habia elegido.

"""
chess_bulk_downloader.py  (versión hardened para run nocturno)
==============================================================
Descarga masiva de 200 partidas por usuario desde Lichess.

PROTECCIONES PARA RUN NOCTURNO:
  ✅ Checkpoint por usuario  → relanzable sin perder trabajo
  ✅ Escritura atómica       → CSV nunca queda corrupto ante corte de luz
  ✅ Stockfish health-check  → detecta cuelgues y reinicia el motor
  ✅ Rate-limit 429          → lee el header Retry-After y espera exacto
  ✅ Backup automático       → copia de seguridad del CSV cada N usuarios
  ✅ Deduplicación doble     → en memoria (set) y al escribir (drop_duplicates)
  ✅ Logging detallado       → bulk_download.log para auditar lo que pasó
  ✅ Guardado de emergencia  → si falla el write principal, salva los datos

"""

import io
import os
import re
import time
import shutil
import pickle
import logging
import tempfile
import traceback
import chess
import chess.pgn
import pandas as pd
import numpy as np
import berserk
from stockfish import Stockfish
from tqdm import tqdm
from datetime import datetime
from pathlib import Path
import os


# ── PATHS ABSOLUTOS ──────────────────────────────────────────────────────
BASE_DIR = r"."

STOCKFISH_PATH   = r"stockfish.exe"
PKL_PATH         = os.path.join(BASE_DIR, "theory_db.pkl")
LICHESS_TOKEN    = r"TU_LICHESS_TOKEN_AQUI"
FILENAME_ML      = os.path.join(BASE_DIR, "master_dataset_ml.csv")
OUTPUT_FILE      = os.path.join(BASE_DIR, "master_dataset_ml.csv")
CHECKPOINT_FILE  = os.path.join(BASE_DIR, "bulk_checkpoint.txt")
LOG_FILE         = os.path.join(BASE_DIR, "bulk_download.log")
BACKUP_DIR       = os.path.join(BASE_DIR, "backups")

GAMES_PER_USER     = 200
ANALYSIS_WINDOW    = 12
STOCKFISH_DEPTH    = 14
RATE_LIMIT_SLEEP   = 1.5
ERROR_SLEEP        = 15.0
MAX_RETRIES        = 3
BACKUP_EVERY_N     = 10
SF_HEALTH_EVERY_N  = 20
# ─────────────────────────────────────────────────────────────────────────
# ═══════════════════════════════════════════════════════════════


# ── LOGGING ─────────────────────────────────────────────────────
os.makedirs(BACKUP_DIR, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.FileHandler(LOG_FILE, encoding="utf-8"),
        logging.StreamHandler(),
    ],
)
log = logging.getLogger(__name__)


# ════════════════════════════════════════════════════════════════
# HELPERS GENERALES
# ════════════════════════════════════════════════════════════════

def acpl_to_accuracy(acpl: float) -> float:
    return round(float(max(0, min(100, 100 * np.exp(-0.0035 * acpl)))), 1)


def load_checkpoint() -> set:
    """Usuarios que ya fueron procesados en sesiones anteriores."""
    if not os.path.exists(CHECKPOINT_FILE):
        return set()
    with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
        return {line.strip() for line in f if line.strip()}


def save_checkpoint(username: str) -> None:
    with open(CHECKPOINT_FILE, "a", encoding="utf-8") as f:
        f.write(username + "\n")


def get_user_list_from_dataset(filename: str) -> list:
    df = pd.read_csv(filename)
    user_ratings = (
        df.groupby("Usuario")["Rating_Usuario"]
        .first()
        .reset_index()
        .sort_values("Rating_Usuario")
    )
    return user_ratings["Usuario"].tolist()


# ════════════════════════════════════════════════════════════════
# ESCRITURA ATÓMICA — el CSV nunca queda a medias
# ════════════════════════════════════════════════════════════════

def atomic_csv_write(df: pd.DataFrame, target_path: str) -> None:
    """
    Escribe en un .tmp en el mismo disco y hace os.replace() al final.
    Si el proceso muere durante la escritura, el original queda intacto.
    """
    target = Path(target_path)
    tmp_fd, tmp_path = tempfile.mkstemp(
        dir=target.parent, prefix=".tmp_", suffix=".csv"
    )
    try:
        os.close(tmp_fd)
        df.to_csv(tmp_path, index=False)
        os.replace(tmp_path, target_path)   # operación atómica
    except Exception:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)
        raise


def make_backup(target_path: str, label: str) -> None:
    """Copia timestamped en backups/."""
    if not os.path.exists(target_path):
        return
    ts   = datetime.now().strftime("%Y%m%d_%H%M%S")
    dest = os.path.join(BACKUP_DIR, f"master_dataset_{label}_{ts}.csv")
    shutil.copy2(target_path, dest)
    log.info(f"💾 Backup → {dest}")


# ════════════════════════════════════════════════════════════════
# DEDUPLICACIÓN — doble barrera
# ════════════════════════════════════════════════════════════════

def load_existing_ids(filepath: str) -> set:
    """
    Primera barrera: carga en memoria los Game_IDs ya procesados.
    Evita re-analizar partidas con Stockfish innecesariamente.
    """
    if not os.path.exists(filepath):
        return set()
    df  = pd.read_csv(filepath, usecols=["Game_ID"])
    ids = set(df["Game_ID"].dropna().astype(str).tolist())
    log.info(f"IDs existentes en memoria: {len(ids):,}")
    return ids


def merge_and_dedup(df_new: pd.DataFrame, output_path: str) -> tuple:
    """
    Segunda barrera: merge con el CSV en disco + drop_duplicates.
    Retorna (filas_añadidas, total_filas).
    """
    if os.path.exists(output_path):
        df_hist  = pd.read_csv(output_path)
        before   = len(df_hist)
        df_final = pd.concat([df_hist, df_new], ignore_index=True)
        df_final = df_final.drop_duplicates(subset=["Game_ID"], keep="first")
        added    = len(df_final) - before
    else:
        df_final = df_new.drop_duplicates(subset=["Game_ID"], keep="first")
        added    = len(df_final)

    atomic_csv_write(df_final, output_path)
    return added, len(df_final)


# ════════════════════════════════════════════════════════════════
# STOCKFISH RESILIENTE — con health-check y auto-restart
# ════════════════════════════════════════════════════════════════

class ResilientStockfish:
    """
    Wrapper sobre Stockfish con detección de cuelgues y reinicio automático.

    Problema real: el proceso hijo de Stockfish puede morir silenciosamente
    (OOM, timeout del SO, etc.) y la librería stockfish-python no lo detecta.
    El script quedaría colgado para siempre esperando una respuesta que nunca llega.

    Solución: health-check periódico con una posición conocida y reinicio
    forzado del proceso si no responde correctamente.
    """
    MATE_SCORE = 1000

    def __init__(self):
        self._eval_count    = 0
        self._restart_count = 0
        self.sf = self._new_engine()

    def _new_engine(self) -> Stockfish:
        sf = Stockfish(
            path=STOCKFISH_PATH,
            parameters={"Threads": 4, "Hash": 512},
        )
        sf.set_depth(STOCKFISH_DEPTH)
        log.info(f"Stockfish iniciado (reinicios acumulados: {self._restart_count})")
        return sf

    def _is_alive(self) -> bool:
        """Test rápido: pide eval de la posición inicial y verifica formato."""
        try:
            self.sf.set_fen_position(
                "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1",
                do_validation=False,
            )
            result = self.sf.get_evaluation()
            return isinstance(result, dict) and "value" in result
        except Exception:
            return False

    def restart(self) -> None:
        log.warning("⚠️  Reiniciando proceso Stockfish...")
        try:
            # La librería guarda el proceso en _stockfish
            if hasattr(self.sf, "_stockfish") and self.sf._stockfish:
                self.sf._stockfish.kill()
        except Exception:
            pass
        self._restart_count += 1
        time.sleep(2)
        self.sf = self._new_engine()

    def health_check(self) -> None:
        """Verifica que el motor responde; lo reinicia si no."""
        if not self._is_alive():
            log.error("❌ Stockfish no responde. Intentando reinicio...")
            self.restart()
            if not self._is_alive():
                raise RuntimeError(
                    "Stockfish no pudo reiniciarse tras health-check. Abortando."
                )
            log.info("✅ Stockfish recuperado correctamente.")

    def get_eval(self, fen: str) -> float:
        """Evaluación en cp con recovery automático ante fallos y posiciones corruptas."""
        self._eval_count += 1

        # ── VALIDACIÓN FEN ANTES DE TOCAR STOCKFISH ───────────────
        try:
            board = chess.Board(fen)
            if not board.is_valid():
                log.warning(f"  get_eval: posición inválida, skip. FEN={fen[:50]}")
                return 0.0
            # Verificación adicional: hay algún rey de cada color
            if not board.king(chess.WHITE) or not board.king(chess.BLACK):
                log.warning(f"  get_eval: posición sin rey, skip. FEN={fen[:50]}")
                return 0.0
        except Exception as e:
            log.warning(f"  get_eval: FEN no parseable ({e}), skip. FEN={fen[:50]}")
            return 0.0
        # ──────────────────────────────────────────────────────────

        # Health-check periódico
        if self._eval_count % SF_HEALTH_EVERY_N == 0:
            self.health_check()

        for attempt in range(3):
            try:
                self.sf.set_fen_position(fen, do_validation=False)
                ev = self.sf.get_evaluation()
                if not isinstance(ev, dict) or "type" not in ev or "value" not in ev:
                    log.warning(f"  get_eval: respuesta inesperada de SF={ev}, skip.")
                    self.restart()
                    continue
                if ev["type"] == "mate":
                    return float(
                        self.MATE_SCORE if ev["value"] > 0 else -self.MATE_SCORE
                    )
                return float(
                    max(-self.MATE_SCORE, min(self.MATE_SCORE, ev["value"]))
                )
            except Exception as e:
                log.warning(f"  get_eval fallo (intento {attempt+1}/3): {e}")
                self.restart()

        log.error("  get_eval falló 3 veces seguidas. Retornando 0.0 (safe fallback).")
        return 0.0


# ════════════════════════════════════════════════════════════════
# SISTEMA DE ANÁLISIS
# ════════════════════════════════════════════════════════════════

class ChessIntelligenceSystem:

    def __init__(self, engine: ResilientStockfish):
        self.engine = engine
        self.theory_positions: set = self._load_theory()

    def _load_theory(self) -> set:
        try:
            with open(PKL_PATH, "rb") as f:
                positions = pickle.load(f)
            log.info(f"Base teórica cargada: {len(positions):,} posiciones.")
            return positions
        except FileNotFoundError:
            log.error(f"No se encuentra el PKL en: {PKL_PATH}")
            return set()

    def calculate_player_acl(
        self, game, player_color: chess.Color, start_move: int,
        window: int = ANALYSIS_WINDOW,
    ) -> float | None:
        board  = game.board()
        moves  = list(game.mainline_moves())
        losses = []

        # Avance hasta start_move con protección ante movimientos ilegales
        for i, m in enumerate(moves[:start_move]):
            try:
                if not board.is_legal(m):
                    log.warning(f"  ACL: movimiento ilegal en ply {i}, abortando análisis.")
                    return None
                board.push(m)
            except Exception as e:
                log.warning(f"  ACL: excepción en push ply {i} ({e}), abortando análisis.")
                return None

        end_idx = min(start_move + window, len(moves))
        if end_idx <= start_move:
            return None

        # Evaluación inicial con validación de posición
        try:
            eval_cache = self.engine.get_eval(board.fen())
        except Exception as e:
            log.warning(f"  ACL: fallo eval inicial ({e}), skip.")
            return None

        for i in range(start_move, end_idx):
            move = moves[i]
            is_player_turn = (board.turn == player_color)
            eval_prev      = eval_cache

            try:
                if not board.is_legal(move):
                    log.warning(f"  ACL: movimiento ilegal en ply {i}, cortando ventana.")
                    break
                board.push(move)
            except Exception as e:
                log.warning(f"  ACL: excepción en push ply {i} ({e}), cortando ventana.")
                break

            try:
                eval_cache = self.engine.get_eval(board.fen())
            except Exception as e:
                log.warning(f"  ACL: fallo eval ply {i} ({e}), cortando ventana.")
                break

            if is_player_turn:
                side = 1.0 if player_color == chess.WHITE else -1.0
                loss = float(max(0.0, (eval_prev - eval_cache) * side))
                losses.append(loss)

        return sum(losses) / len(losses) if losses else None

    def analyze_games(
        self, games_list: list, target_user: str,
        user_rating: int, known_ids: set,
    ) -> pd.DataFrame:
        results = []
        for game in games_list:
            if not game:
                continue

            headers  = game.headers
            game_id  = headers.get("Site", "").split("/")[-1]

            # Verificación de duplicado a nivel de partida individual
            if game_id in known_ids:
                continue

            white_player  = headers.get("White", "").lower()
            is_white      = (white_player == target_user.lower())
            user_color    = chess.WHITE if is_white else chess.BLACK

            apertura_full = headers.get("Opening", "Desconocida")
            apertura_base = apertura_full.split(":")[0].split(",")[0].strip()

            res_str = headers.get("Result", "*")
            is_win  = (
                1.0  if (res_str == "1-0" and is_white) or (res_str == "0-1" and not is_white)
                else 0.5 if res_str == "1/2-1/2"
                else 0.0
            )

            board      = game.board()
            moves      = list(game.mainline_moves())
            theory_end = 0
            game_corrupt = False
            for i, move in enumerate(moves[:25]):
                try:
                    if not board.is_legal(move):
                        log.warning(f"  Theory: movimiento ilegal ply {i} en {game_id}, truncando.")
                        break
                    board.push(move)
                except Exception as e:
                    log.warning(f"  Theory: posición corrupta ply {i} en {game_id} ({e}), skip partida.")
                    game_corrupt = True
                    break
                if board.epd() in self.theory_positions:
                    theory_end = i + 1
                else:
                    break
            if game_corrupt:
                continue

            try:
                acl = self.calculate_player_acl(game, user_color, theory_end)
            except Exception as e:
                log.warning(f"  ACL falló para {game_id}: {e}. Saltando partida.")
                continue

            if acl is not None:
                results.append({
                    "Game_ID":        game_id,
                    "Usuario":        target_user,
                    "Rating_Usuario": user_rating,
                    "Apertura":       apertura_base,
                    "Color":          "Blancas" if is_white else "Negras",
                    "Fin_Teoria":     theory_end,
                    "ACL_Post_Teo":   round(acl, 1),
                    "Victoria":       is_win,
                })
                # Registra inmediatamente para evitar duplicados en el mismo stream
                known_ids.add(game_id)

        return pd.DataFrame(results)


# ════════════════════════════════════════════════════════════════
# MANEJO DE RATE LIMIT 429 DE LICHESS
# ════════════════════════════════════════════════════════════════

def lichess_call(fn, *args, **kwargs):
    """
    Ejecuta fn con reintentos y manejo específico del 429.
    Lichess devuelve un header Retry-After que indica exactamente
    cuántos segundos esperar — lo usamos en lugar de un backoff fijo.
    """
    for attempt in range(MAX_RETRIES):
        try:
            return fn(*args, **kwargs)

        except berserk.exceptions.ResponseError as e:
            # Extraer status_code del atributo o del mensaje
            status = getattr(e, "status_code", 0)
            if not status:
                match = re.search(r"(\d{3})", str(e))
                status = int(match.group(1)) if match else 0

            if status == 429:
                retry_after = 60  # default conservador
                if hasattr(e, "response") and e.response is not None:
                    retry_after = int(
                        e.response.headers.get("Retry-After", 60)
                    )
                wait = retry_after + 5  # +5s de margen
                log.warning(
                    f"  ⏳ Rate limit 429. Esperando {wait}s "
                    f"(intento {attempt+1}/{MAX_RETRIES})..."
                )
                time.sleep(wait)
            else:
                log.error(f"  Error API {status} (intento {attempt+1}): {e}")
                if attempt < MAX_RETRIES - 1:
                    time.sleep(ERROR_SLEEP * (attempt + 1))
                else:
                    raise

        except Exception as e:
            log.error(f"  Error inesperado (intento {attempt+1}): {e}")
            if attempt < MAX_RETRIES - 1:
                time.sleep(ERROR_SLEEP * (attempt + 1))
            else:
                raise

    return None


# ════════════════════════════════════════════════════════════════
# PROCESADO DE UN USUARIO
# ════════════════════════════════════════════════════════════════

def procesar_usuario(
    username: str,
    client: berserk.Client,
    system: ChessIntelligenceSystem,
    existing_ids: set,
) -> int:
    log.info(f"─── [{username}] ────────────────────────────────────")

    # 1. Perfil y rating
    try:
        perfil  = lichess_call(client.users.get_public_data, username)
        if perfil is None:
            log.warning(f"[{username}] No se pudo obtener el perfil.")
            return 0
        perfs   = perfil.get("perfs", {})
        r_blitz = perfs.get("blitz", {}).get("rating", 0)
        r_rapid = perfs.get("rapid", {}).get("rating", 0)
        rating  = r_blitz if r_blitz > 0 else r_rapid

        if rating == 0:
            log.warning(f"[{username}] Sin rating activo. Saltando.")
            return 0
        log.info(f"[{username}] Blitz={r_blitz} | Rapid={r_rapid} → ref={rating}")

    except Exception as e:
        log.error(f"[{username}] Fallo al obtener perfil: {e}")
        return 0

    # 2. Descarga del stream PGN
    time.sleep(RATE_LIMIT_SLEEP)
    try:
        games_gen = lichess_call(
            client.games.export_by_player,
            username,
            max=GAMES_PER_USER,
            opening=True,
            as_pgn=True,
        )
        if games_gen is None:
            log.error(f"[{username}] export_by_player devolvió None.")
            return 0
    except Exception as e:
        log.error(f"[{username}] No se pudo iniciar la descarga: {e}")
        return 0

    # Parseo del stream — robusto ante elementos corruptos
    games = []
    try:
        for g_str in games_gen:
            if isinstance(g_str, dict):
                continue
            try:
                game_obj = chess.pgn.read_game(io.StringIO(g_str))
            except Exception:
                continue
            if not game_obj:
                continue
            gid = game_obj.headers.get("Site", "").split("/")[-1]
            if gid not in existing_ids:
                games.append(game_obj)
    except Exception as e:
        log.error(f"[{username}] Error durante parseo del stream: {e}")
        # Continúa con lo que haya llegado a parsear

    log.info(f"[{username}] {len(games)} partidas nuevas encontradas.")
    if not games:
        log.info(f"[{username}] Nada nuevo que analizar.")
        return 0

    # 3. Análisis Stockfish
    try:
        df_new = system.analyze_games(games, username, rating, existing_ids)
    except Exception as e:
        log.error(
            f"[{username}] Fallo en analyze_games: {e}\n{traceback.format_exc()}"
        )
        return 0

    if df_new.empty:
        log.warning(f"[{username}] El análisis no produjo resultados.")
        return 0

    # 4. Persistencia atómica + deduplicación en disco
    try:
        added, total = merge_and_dedup(df_new, OUTPUT_FILE)
        log.info(f"[{username}] ✅ +{added} filas | Total dataset: {total:,}")
        return added

    except Exception as e:
        log.error(
            f"[{username}] Fallo al guardar CSV: {e}\n{traceback.format_exc()}"
        )
        # Guardado de emergencia: los datos NO se pierden aunque el CSV principal falle
        ts_emerg   = datetime.now().strftime("%H%M%S")
        emerg_file = f"emergency_{username}_{ts_emerg}.csv"
        df_new.to_csv(emerg_file, index=False)
        log.error(f"[{username}] ⚠️  Datos guardados en emergencia: {emerg_file}")
        return 0


# ════════════════════════════════════════════════════════════════
# MAIN
# ════════════════════════════════════════════════════════════════

def main():
    start_time = datetime.now()
    log.info("═" * 62)
    log.info("  CHESS BULK DOWNLOADER — RUN NOCTURNO (hardened)")
    log.info(f"  Inicio: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
    log.info(f"  Objetivo: {GAMES_PER_USER} partidas/usuario")
    log.info("═" * 62)

    # Validaciones de arranque
    checks = {
        f"Dataset '{FILENAME_ML}'":          os.path.exists(FILENAME_ML),
        f"Stockfish '{STOCKFISH_PATH}'":      os.path.exists(STOCKFISH_PATH),
        f"Teoría PKL '{PKL_PATH}'":           os.path.exists(PKL_PATH),
    }
    abort = False
    for desc, ok in checks.items():
        status = "✅" if ok else "❌ NO ENCONTRADO"
        log.info(f"  {status}  {desc}")
        if not ok:
            abort = True
    if abort:
        log.error("Abortando — corrige los paths antes de continuar.")
        return

    # Lista de pendientes
    user_list  = get_user_list_from_dataset(FILENAME_ML)
    done_users = load_checkpoint()
    pending    = [u for u in user_list if u not in done_users]

    log.info(f"Usuarios totales:  {len(user_list)}")
    log.info(f"Ya procesados:     {len(done_users)}")
    log.info(f"Pendientes:        {len(pending)}")

    if not pending:
        log.info("✅ Todos los usuarios ya están al día.")
        return

    # Carga de IDs en memoria (primera barrera de deduplicación)
    existing_ids = load_existing_ids(FILENAME_ML)

    # Backup previo al run
    make_backup(FILENAME_ML, "pre_run")

    # Inicializar motor y cliente
    engine = ResilientStockfish()
    engine.health_check()   # verifica respuesta antes de empezar el bucle

    system = ChessIntelligenceSystem(engine)
    client = berserk.Client(berserk.TokenSession(LICHESS_TOKEN))

    # Bucle principal
    total_added  = 0
    users_done   = 0
    failed_users = []

    with tqdm(pending, desc="Usuarios", unit="user", colour="cyan") as pbar:
        for username in pbar:
            pbar.set_postfix({
                "user":     username[:14],
                "añadidas": total_added,
                "SF_reini": engine._restart_count,
            })

            try:
                added = procesar_usuario(username, client, system, existing_ids)
                total_added += added
                if added == 0:
                    failed_users.append(username)
            except Exception as e:
                log.error(
                    f"[{username}] Error no capturado: {e}\n{traceback.format_exc()}"
                )
                failed_users.append(username)

            # Checkpoint siempre, incluso si added == 0
            save_checkpoint(username)
            users_done += 1

            # Backup periódico
            if users_done % BACKUP_EVERY_N == 0:
                make_backup(OUTPUT_FILE, f"mid_{users_done}u")

            time.sleep(RATE_LIMIT_SLEEP)

    # Backup final
    make_backup(OUTPUT_FILE, "post_run")

    # ── Resumen ──────────────────────────────────────────────────
    elapsed  = (datetime.now() - start_time).total_seconds() / 60
    df_final = pd.read_csv(OUTPUT_FILE)

    log.info("═" * 62)
    log.info("  PROCESO COMPLETADO")
    log.info(f"  Duración:            {elapsed:.1f} min")
    log.info(f"  Partidas añadidas:   {total_added:,}")
    log.info(f"  Total en dataset:    {len(df_final):,}")
    log.info(f"  Usuarios únicos:     {df_final['Usuario'].nunique()}")
    log.info(f"  Reinicios Stockfish: {engine._restart_count}")
    if failed_users:
        log.warning(f"  Usuarios sin datos:  {len(failed_users)}")
        log.warning(f"  → {failed_users}")
    log.info("═" * 62)

    # ── Distribución final ───────────────────────────────────────
    bins   = [0,1000,1200,1400,1600,1800,2000,2200,2400,5000]
    labels = ["<1000","1000-1200","1200-1400","1400-1600","1600-1800",
              "1800-2000","2000-2200","2200-2400",">2400"]
    df_final["Tramo"] = pd.cut(df_final["Rating_Usuario"], bins=bins, labels=labels)
    dist = df_final.groupby("Tramo", observed=True).agg(
        Usuarios = ("Usuario",      "nunique"),
        Partidas = ("Game_ID",      "count"),
        Teo_Med  = ("Fin_Teoria",   lambda x: round(x.mean(), 1)),
        Acc_Med  = ("ACL_Post_Teo", "mean"),
    )
    dist["Acc%"] = dist["Acc_Med"].apply(acpl_to_accuracy)
    print("\n📊 DISTRIBUCIÓN FINAL DEL DATASET:")
    print(dist[["Usuarios", "Partidas", "Teo_Med", "Acc%"]].to_string())

    # ── Validación final de duplicados ───────────────────────────
    n_dups = df_final.duplicated(subset=["Game_ID"]).sum()
    if n_dups > 0:
        log.warning(f"⚠️  {n_dups} duplicados residuales detectados. Limpiando...")
        df_clean = df_final.drop_duplicates(subset=["Game_ID"], keep="first")
        atomic_csv_write(df_clean, OUTPUT_FILE)
        log.info(f"✅ Duplicados eliminados. Filas finales: {len(df_clean):,}")
    else:
        log.info(f"✅ Cero duplicados. Dataset 100% limpio.")


if __name__ == "__main__":
    main()